# Прогноз розничного товарооборота с пространственными признаками

Ноутбук воспроизводит итоговый прогноз розничного товарооборота магазинов сети «Пятерочка» на
март 2025 года, получивший 90.83 балла на полном тесте соревнования (исходный ансамбль - 90.78).

Исходное решение - ансамбль четырех градиентных бустингов: CatBoost и LightGBM на логарифме
выручки, LightGBM и XGBoost на логарифмическом приросте. Код признаков, параметры и правило
объединения прогнозов перенесены из исходного ноутбука без изменений. Пространственная
корректировка следует модели SLX: к признакам добавляются их пространственные лаги $WX$ по
соседним населенным пунктам. Матрица $W$ - симметризованная матрица $k$ ближайших населенных
пунктов в проекции Альберса с построчной нормировкой.

Итоговый состав ансамбля:

- CatBoost - исходная версия без $WX$;
- LightGBM на приростах - $WX$ по матрице kNN12;
- LightGBM на уровнях - $WX$ по матрице kNN6;
- XGBoost на приростах - $WX$ по матрице kNN12.

Все модели обучаются на январе 2023 - феврале 2025 года с исходным полным числом деревьев и без
ранней остановки. Входные файлы `train_2.csv`, `settlement_coordinates.csv` и
`settlement_panel.parquet` ищутся в папке `Data` рядом с ноутбуком, а если их там нет - в
структуре проекта.
Нужны пакеты catboost, lightgbm, xgboost, holidays, pyproj, scipy, pandas и pyarrow. Исходный XGBoost обучен на CUDA с xgboost 3.1.3; на
CPU его прогноз может немного отличаться.

In [ ]:
import gc
import hashlib
import json
import os
import shutil
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import holidays
from scipy import sparse
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

NOTEBOOK_DIRECTORY = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIRECTORY if (NOTEBOOK_DIRECTORY / "Data").exists() else NOTEBOOK_DIRECTORY.parent


def input_path(file_name, project_location):
    """Файл в папке Data рядом с ноутбуком или, если его там нет, в структуре проекта."""
    local = NOTEBOOK_DIRECTORY / "Data" / file_name
    return local if local.exists() else PROJECT_ROOT / project_location


SHOPS_PATH = input_path("train_2.csv", "Data/Shops/train_2.csv")
COORDINATES_PATH = input_path("settlement_coordinates.csv",
                              "Parsing/geonames_settlement_coordinates/result/settlement_coordinates.csv")
SETTLEMENT_PANEL_PATH = input_path("settlement_panel.parquet", "Data/settlement_panel.parquet")
SUBMISSION_PATH = NOTEBOOK_DIRECTORY / "forecast_march_2025.csv"

MODELS = ["catboost_log_level", "lightgbm_log_diff", "lightgbm_log_level", "xgboost_log_diff"]
# Полное число деревьев исходного решения
TREE_COUNTS = {"catboost_log_level": 6500, "lightgbm_log_diff": 7000,
               "lightgbm_log_level": 8000, "xgboost_log_diff": 3750}
# Итоговый состав: матрица весов для WX или исходная версия без WX
FINAL_CONFIGURATION = {"catboost_log_level": "baseline", "lightgbm_log_diff": "wx_knn12",
                       "lightgbm_log_level": "wx_knn6", "xgboost_log_diff": "wx_knn12"}
# Короткий проверочный прогон: FORECAST_TREE_LIMIT ограничивает число деревьев
TREE_LIMIT = int(os.environ.get("FORECAST_TREE_LIMIT", "0"))
if TREE_LIMIT > 0:
    TREE_COUNTS = {model: min(count, TREE_LIMIT) for model, count in TREE_COUNTS.items()}
THREADS = int(os.environ.get("FORECAST_THREADS", "4"))
XGBOOST_DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("Деревья:", TREE_COUNTS, " устройство XGBoost:", XGBOOST_DEVICE)

## Исходное решение

Ячейки ниже перенесены из исходного ноутбука без изменений: переименование столбцов, генерация
признаков и разбиение, отобранные признаки и гиперпараметры четырех моделей, вспомогательные
функции подготовки данных и функции обучения CatBoost, LightGBM и XGBoost.

### Переименование столбцов

In [ ]:
rename_cols = {
    "new_id": "new_id",
    "Год": "year",
    "Месяц": "month",
    "Среднее количество промо товаров в чеке": "avg_promo_items_per_receipt",
    "Среднее количество товаров в чеке": "avg_items_per_receipt",
    "Среднее количество отмен": "avg_cancellations",
    "Рабочие часы в день": "working_hours_per_day",
    "Дата открытия, категориальный": "opening_date_category",
    "Торговая площадь, категориальный": "trading_area_category",
    "Населенный пункт": "settlement",
    "Регион": "region",
    "Численность населения": "population_size",
    "Количество домохозяйств": "number_of_households",
    "Трафик пеший, в час": "pedestrian_traffic_per_hour",
    "Трафик авто, в час": "car_traffic_per_hour",
    "Маркетплейсы, доставки, постаматы (100 м)": "marketplaces_deliveries_parcel_lockers_100m",
    "Медицинские уч. и аптеки (300 м)": "medical_institutions_and_pharmacies_300m",
    "Школы (300 м)": "schools_300m",
    "Остановки (300 м)": "public_transport_stops_300m",
    "Продуктовые магазины (500 м)": "grocery_stores_500m",
    "Пятерочки (500 м)": "pyaterochka_stores_500m",
    "Количество касс": "number_of_cash_registers",
    "Флаг алкогольной лицензии": "alcohol_license_flag",
    "РТО": "pto",
}

### Генерация признаков и разбиение

In [ ]:
from typing import Literal, Optional, Sequence

import gc
import warnings
import numpy as np
import pandas as pd


def make_panel_time_split(
    df: pd.DataFrame,
    *,
    group_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    min_year: Optional[int] = None,
    create_time_col: bool = True,
    create_test_if_missing: bool = True,
    test_target_value=np.nan,
    future_unknown_cols: Optional[Sequence[str]] = None,
    drop_target_na_from_train: bool = True,
    require_known_val_target: bool = True,
    check_duplicates: bool = True,
    validate_time_col: bool = True,
    expected_test_size: Optional[int] = None,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Leakage-safe time split для panel monthly forecasting.

    split_mode:
    - "train_val": train = months before val, val = selected validation month;
    - "train_val_test": train + val + artificial/existing test month;
    - "train_test": train = all known history before test, test = artificial/existing test month.

    Если test-месяца нет в df, он создаётся из последней известной строки каждого магазина.
    Важно: если дальше строятся лаги/rolling, test-строки должны быть созданы ДО feature engineering.
    """
    if split_mode not in {"train_val", "train_val_test", "train_test"}:
        raise ValueError("split_mode должен быть 'train_val', 'train_val_test' или 'train_test'.")

    if copy:
        df = df.copy()

    def _check_time_args(prefix, time_idx, year, month):
        if time_idx is not None and (year is not None or month is not None):
            raise ValueError(
                f"Для {prefix} передай либо {prefix}_time_idx, либо пару "
                f"{prefix}_year + {prefix}_month, но не оба варианта."
            )
        if (year is None) ^ (month is None):
            raise ValueError(
                f"Для {prefix} нужно передавать year и month вместе. "
                f"Получено: {prefix}_year={year}, {prefix}_month={month}."
            )

    _check_time_args("val", val_time_idx, val_year, val_month)
    _check_time_args("test", test_time_idx, test_year, test_month)

    if split_mode == "train_test" and (
        val_time_idx is not None or val_year is not None or val_month is not None
    ):
        raise ValueError("В split_mode='train_test' validation-месяц не используется.")

    required_cols = {group_col, target_col}
    has_time_col = time_col in df.columns
    has_year_month = year_col in df.columns and month_col in df.columns

    if not has_time_col:
        if create_time_col:
            required_cols.update({year_col, month_col})
        else:
            required_cols.add(time_col)

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"В df отсутствуют обязательные колонки: {sorted(missing)}")

    if has_year_month:
        if df[year_col].isna().any() or df[month_col].isna().any():
            raise ValueError(f"В {year_col}/{month_col} есть пропуски.")
        df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
        df[month_col] = pd.to_numeric(df[month_col], errors="raise").astype(int)
        bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
        if bad_months:
            raise ValueError(f"В колонке '{month_col}' некорректные месяцы: {bad_months}.")

    if has_time_col:
        if df[time_col].isna().any():
            raise ValueError(f"В колонке '{time_col}' есть пропуски.")
        df[time_col] = pd.to_numeric(df[time_col], errors="raise").astype(int)

        if min_year is None and has_year_month:
            inferred = df[year_col] - ((df[time_col] - (df[month_col] - 1)) // 12)
            unique = sorted(inferred.unique())
            if len(unique) == 1:
                min_year = int(unique[0])
            else:
                raise ValueError(
                    f"Не удалось однозначно восстановить min_year из {time_col}. "
                    f"Передай min_year явно. Найдено: {unique[:10]}"
                )

        if validate_time_col and has_year_month and min_year is not None:
            expected = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)
            bad = df[time_col].astype(int) != expected
            if bad.any():
                examples = df.loc[bad, [group_col, year_col, month_col, time_col]].head(10)
                raise ValueError(
                    f"'{time_col}' не совпадает с расчётом по min_year={min_year}. "
                    f"Примеры:\n{examples}"
                )
    else:
        if not has_year_month:
            raise ValueError(f"Нельзя создать '{time_col}', потому что нет year/month.")
        if min_year is None:
            min_year = int(df[year_col].min())
        df[time_col] = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)

    known_target_times = sorted(df.loc[df[target_col].notna(), time_col].unique())
    if not known_target_times:
        raise ValueError(f"В '{target_col}' нет известных значений.")
    max_known_time_idx = int(known_target_times[-1])

    def _resolve(label, time_idx, year, month, default=None):
        if time_idx is not None:
            return int(time_idx)
        if year is not None and month is not None:
            if min_year is None:
                raise ValueError(f"Для расчёта {label}_time_idx нужен min_year.")
            month = int(month)
            if not 1 <= month <= 12:
                raise ValueError(f"{label}_month должен быть от 1 до 12.")
            return (int(year) - int(min_year)) * 12 + (month - 1)
        return None if default is None else int(default)

    if split_mode in {"train_val", "train_val_test"}:
        val_time_idx = _resolve("val", val_time_idx, val_year, val_month, max_known_time_idx)
    else:
        val_time_idx = None

    if split_mode in {"train_val_test", "train_test"}:
        test_time_idx = _resolve(
            "test", test_time_idx, test_year, test_month, max_known_time_idx + 1
        )
    else:
        test_time_idx = None

    if split_mode == "train_val_test" and not (val_time_idx < test_time_idx):
        raise ValueError(
            f"Ожидается val_time_idx < test_time_idx, получено {val_time_idx=} и {test_time_idx=}."
        )

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)

    if check_duplicates:
        dup_mask = df.duplicated([group_col, time_col], keep=False)
        if dup_mask.any():
            examples = (
                df.loc[dup_mask, [group_col, time_col]].sort_values([group_col, time_col]).head(10)
            )
            raise ValueError(
                f"Есть дубли по [{group_col}, {time_col}]. "
                f"Строк-дублей: {int(dup_mask.sum())}. Примеры:\n{examples}"
            )

    test = None
    if split_mode in {"train_val_test", "train_test"}:
        existing = df[df[time_col] == test_time_idx].copy()
        if not existing.empty:
            test = existing.copy()
        else:
            if not create_test_if_missing:
                raise ValueError(f"В df нет test_time_idx={test_time_idx}.")
            hist = df[(df[time_col] < test_time_idx) & (df[target_col].notna())].copy()
            if hist.empty:
                raise ValueError(f"Нет исторических строк до test_time_idx={test_time_idx}.")
            test = (
                hist.sort_values([group_col, time_col])
                .groupby(group_col, sort=False)
                .tail(1)
                .copy()
            )
            test[time_col] = int(test_time_idx)
            if has_year_month and min_year is not None:
                test[year_col] = int(min_year) + int(test_time_idx) // 12
                test[month_col] = int(test_time_idx) % 12 + 1
            test[target_col] = test_target_value

            if future_unknown_cols is not None:
                for col in future_unknown_cols:
                    if col in test.columns:
                        test[col] = np.nan
                    else:
                        warnings.warn(f"Колонка '{col}' из future_unknown_cols отсутствует в df.")

        test = test.sort_values([group_col, time_col]).reset_index(drop=True)
        if expected_test_size is not None and len(test) != int(expected_test_size):
            raise ValueError(
                f"Размер test не совпадает с expected_test_size: "
                f"получено {len(test)}, ожидалось {expected_test_size}."
            )

    if split_mode == "train_test":
        train_mask = df[time_col] < test_time_idx
    else:
        train_mask = df[time_col] < val_time_idx

    if drop_target_na_from_train:
        train_mask = train_mask & df[target_col].notna()

    train = df.loc[train_mask].copy()
    if train.empty:
        raise ValueError("train получился пустым.")

    val = None
    if split_mode in {"train_val", "train_val_test"}:
        val = df[df[time_col] == val_time_idx].copy()
        if val.empty:
            available = sorted(df[time_col].unique())
            raise ValueError(
                f"val пустой для val_time_idx={val_time_idx}. "
                f"Доступные time_idx: {available[:5]} ... {available[-5:]}"
            )
        if require_known_val_target and val[target_col].isna().any():
            n_nan = int(val[target_col].isna().sum())
            raise ValueError(
                f"В validation есть NaN в '{target_col}': {n_nan}. "
                "Скорее всего, validation попал на искусственный test-месяц."
            )
        val = val.sort_values([group_col, time_col]).reset_index(drop=True)

    train = train.sort_values([group_col, time_col]).reset_index(drop=True)

    if verbose:
        print("split_mode:", split_mode)
        print("min_year:", min_year)
        print("max_known_time_idx:", max_known_time_idx)
        if val_time_idx is not None:
            print("val_time_idx:", val_time_idx)
        if test_time_idx is not None:
            print("test_time_idx:", test_time_idx)
        print("train shape:", train.shape)
        if val is not None:
            print("val shape:", val.shape)
        if test is not None:
            print("test shape:", test.shape)

    if split_mode == "train_val":
        return train, val
    if split_mode == "train_test":
        return train, test
    return train, val, test


# ============================================================
# Base feature generator from uploaded file
# ============================================================

import gc
import numpy as np
import pandas as pd


def generate_retail_forecast_features_leakfree(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame | None = None,
    test_df: pd.DataFrame | None = None,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    make_val_from_train: bool = False,
    val_time_idx: int | None = None,
    use_val_as_test_history: bool = True,
    add_calendar_features: bool = True,
    add_static_features: bool = True,
    add_target_history: bool = True,
    add_category_target_history: bool = True,
    add_non_target_monthly_history: bool = True,
    include_current_receipt_features: bool = False,
    current_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    current_monthly_derived_cols: tuple[str, ...] = (
        "promo_items_share",
        "promo_x_items",
        "items_x_working_hours",
        "items_x_working_days",
        "alcohol_x_items",
    ),
    working_hours_future_known: bool = False,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    target_rolling_windows: tuple[int, ...] = (3, 12),
    target_rolling_min_periods: int = 2,
    target_history_scale: str = "raw",  # raw | log | both
    keep_raw_pto_lag_1_anchor_for_log: bool = True,
    add_log_target_dynamics: bool = True,
    group_specs: list | None = None,
    group_agg_stat: str = "mean",
    group_lag: int = 1,
    add_group_lag_feature: bool = True,
    add_group_relative_ratio: bool = True,
    add_group_relative_diff: bool = False,
    monthly_feature_config: dict | None = None,
    monthly_history_scale: str = "raw",  # raw | log | both
    monthly_rolling_min_periods: int = 2,
    add_monthly_cross_lags: bool = True,
    monthly_cross_lags: tuple[int, ...] = (1, 12),
    monthly_cross_feature_types: tuple[str, ...] = ("promo_share",),
    drop_current_monthly_cols: bool = True,
    drop_current_monthly_derived_cols: bool = True,
    drop_year_col: bool = True,
    drop_obvious_duplicates: bool = True,
    extra_drop_cols: list | tuple | None = None,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Единый leak-free feature-pipeline для panel monthly forecasting.

    Основной контракт:
    - val/test target не используется для лагов и group target history;
    - текущие monthly-cols из val/test не используются для лагов;
    - лаги считаются через time-aware merge по time_col - lag, а не через shift строк;
    - rolling-признаки считаются только по прошлым наблюдениям внутри history;
    - для test можно использовать train+val как историю, если val уже является известным прошлым.
    """

    if target_history_scale not in {"raw", "log", "both"}:
        raise ValueError("target_history_scale должен быть 'raw', 'log' или 'both'.")
    if monthly_history_scale not in {"raw", "log", "both"}:
        raise ValueError("monthly_history_scale должен быть 'raw', 'log' или 'both'.")
    if group_agg_stat not in {"mean", "median"}:
        raise ValueError("group_agg_stat должен быть 'mean' или 'median'.")

    if copy:
        train_df = train_df.copy()
        val_df = val_df.copy() if val_df is not None else None
        test_df = test_df.copy() if test_df is not None else None

    # ============================================================
    # Helpers
    # ============================================================

    def _safe_divide(numerator, denominator, fill=np.nan):
        numerator = numerator if isinstance(numerator, pd.Series) else pd.Series(numerator)
        denominator = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator)
        result = numerator / denominator.replace(0, np.nan)
        result = result.replace([np.inf, -np.inf], np.nan)
        if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
            result = result.fillna(fill)
        return result

    def _col(df, name, default=0.0):
        if name in df.columns:
            return df[name].fillna(default)
        return pd.Series(default, index=df.index)

    def _ensure_time_year_month(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
        df = df.copy()

        has_year_month = year_col in df.columns and month_col in df.columns
        has_time = time_col in df.columns

        if not has_year_month and not has_time:
            raise ValueError(f"В {name} нет ни пары '{year_col}'+'{month_col}', ни '{time_col}'.")

        if has_year_month:
            if df[year_col].isna().any() or df[month_col].isna().any():
                raise ValueError(f"В {name} есть NA в {year_col}/{month_col}.")
            df[year_col] = df[year_col].astype(int)
            df[month_col] = df[month_col].astype(int)
            bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
            if bad_months:
                raise ValueError(f"В {name} некорректные месяцы: {bad_months}.")
            # Важно: всегда пересчитываем глобальную шкалу, чтобы не было local-min-year.
            df[time_col] = ((df[year_col] - min_year) * 12 + (df[month_col] - 1)).astype(int)
        else:
            if df[time_col].isna().any():
                raise ValueError(f"В {name} есть NA в {time_col}.")
            df[time_col] = df[time_col].astype(int)
            df[year_col] = (min_year + df[time_col] // 12).astype(int)
            df[month_col] = (df[time_col] % 12 + 1).astype(int)

        return df

    def _validate_required(df: pd.DataFrame, required: set[str], name: str):
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"В {name} нет обязательных колонок: {missing}")

    def _validate_unique_panel(df: pd.DataFrame, name: str):
        if {id_col, time_col}.issubset(df.columns):
            duplicated = df.duplicated([id_col, time_col]).sum()
            if duplicated > 0:
                raise ValueError(
                    f"В {name} найдено {duplicated} дублей по [{id_col}, {time_col}]. "
                    "Для лагов ожидается одна строка на магазин-месяц."
                )

    def _part_concat(parts: dict[str, pd.DataFrame]) -> pd.DataFrame:
        out = []
        for part_name, df_part in parts.items():
            p = df_part.copy()
            p["__part__"] = part_name
            p["__row_order__"] = np.arange(len(p))
            out.append(p)
        return pd.concat(out, axis=0, ignore_index=True, sort=False)

    def _split_parts(combined: pd.DataFrame, part_names: list[str]) -> dict[str, pd.DataFrame]:
        result = {}
        for part_name in part_names:
            part = (
                combined[combined["__part__"] == part_name]
                .sort_values("__row_order__")
                .drop(columns=["__part__", "__row_order__"], errors="ignore")
                .reset_index(drop=True)
            )
            result[part_name] = part
        return result

    def _drop_preexisting_generated_cols(df: pd.DataFrame) -> pd.DataFrame:
        prefixes = [
            f"{target_col}_lag_",
            f"log_{target_col}_lag_",
            f"{target_col}_roll_",
            f"{target_col}_diff_",
            f"{target_col}_ratio_",
            f"{target_col}_seasonal_",
            f"log_{target_col}_diff_",
            f"{target_col}_lag_1_to_",
            f"{target_col}_lag_12_to_",
            "promo_items_share_lag_",
            "promo_x_items_lag_",
            "items_x_working_hours_lag_",
            "cancellations_per_item_lag_",
            "cancellations_per_working_hour_lag_",
            "region_",
            "settlement_",
        ]
        monthly_prefixes = []
        for c in current_monthly_cols:
            monthly_prefixes.extend(
                [
                    f"{c}_lag_",
                    f"log_{c}_lag_",
                    f"{c}_roll_",
                    f"{c}_diff_",
                    f"{c}_ratio_",
                ]
            )

        explicit = {
            "season",
            "is_winter",
            "is_spring",
            "is_autumn",
            "is_quarter_start_month",
            "is_quarter_end_month",
            "is_halfyear_start_month",
            "is_halfyear_end_month",
            "is_year_start_month",
            "is_year_end_month",
            "month_in_halfyear",
            "month_name",
            "month_sin",
            "month_cos",
            "month_sin_2",
            "month_cos_2",
            "prev_month",
            "n_days_in_month",
            "n_weekend_days",
            "n_may_holiday_days",
            "n_single_public_holiday_days",
            "n_november_holiday_days",
            "n_december_additional_holiday_days",
            "n_holiday_days",
            "n_additional_holiday_days",
            "n_working_days",
            "n_additional_working_days",
            "is_leap_year",
            "holiday_month_type",
            "is_middle_of_quarter",
            "is_pre_new_year_month",
            "is_pre_may_holidays_month",
            "is_back_to_school_month",
            "is_business_activity_low_month",
            "is_business_activity_high_month",
            "total_traffic_per_hour",
            "pedestrian_traffic_share",
            "car_to_pedestrian_traffic_ratio",
            "log_population_size",
            "log_number_of_households",
            "log_pedestrian_traffic_per_hour",
            "log_car_traffic_per_hour",
            "is_moscow",
            "is_million_plus_city",
            "population_city_type",
            "has_school_nearby",
            "has_transport_nearby",
            "has_marketplace_nearby",
            "has_medical_nearby",
            "has_grocery_competition",
            "has_pyaterochka_nearby",
            "is_high_grocery_competition",
            "is_high_pyaterochka_density",
            "non_pyaterochka_grocery_stores_500m",
            "pyaterochka_share_among_grocery",
            "non_pyaterochka_competition_share",
            "grocery_per_1000_people",
            "pyaterochka_per_1000_people",
            "competition_per_1000_households",
            "pyaterochka_per_1000_households",
            "people_per_household",
            "households_per_1000_people",
            "traffic_per_1000_people",
            "traffic_per_1000_households",
            "pedestrian_traffic_per_1000_people",
            "social_infra_score",
            "commercial_infra_score",
            "poi_score",
            "poi_per_1000_households",
            "public_transport_share_in_social_infra",
            "medical_share_in_social_infra",
            "cash_register_hours_capacity",
            "traffic_per_cash_register",
            "traffic_per_working_hour",
            "households_per_cash_register",
            "population_per_cash_register",
            "cash_registers_per_total_traffic",
            "promo_items_share",
            "promo_x_items",
            "items_x_working_hours",
            "items_x_working_days",
            "alcohol_x_capacity",
            "alcohol_x_total_traffic",
            "alcohol_x_items",
            "traffic_x_total_poi",
            "traffic_x_grocery_competition",
            "traffic_x_pyaterochka_competition",
            "pedestrian_traffic_x_transport",
            "schools_x_back_to_school",
            "settlement_store_count",
            "region_store_count",
            "region_trading_area_store_count",
            "region_opening_age_store_count",
        }

        drop_cols = []
        for c in df.columns:
            if c in explicit:
                drop_cols.append(c)
            elif any(c.startswith(p) for p in prefixes + monthly_prefixes):
                # Не удаляем сырые region/settlement, только generated mean/count; это грубый фильтр ниже уточняем.
                if c in {"region", "settlement"}:
                    continue
                # region_* может быть исходной колонкой только маловероятно; для безопасности удаляем только *_mean/count.
                if c.startswith(("region_", "settlement_")) and not (
                    c.endswith("_mean") or c.endswith("_count")
                ):
                    continue
                drop_cols.append(c)
            elif f"_{target_col}_{group_agg_stat}_lag_" in c:
                drop_cols.append(c)

        return df.drop(columns=sorted(set(drop_cols)), errors="ignore")

    # ============================================================
    # Calendar and static feature blocks
    # ============================================================

    def _add_calendar(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        years = sorted(df[year_col].dropna().astype(int).unique())
        if not years:
            return df

        def _holiday_calendar():
            holidays = {
                # 2023
                "2023-01-01": "New_Year_Holidays",
                "2023-01-02": "New_Year_Holidays",
                "2023-01-03": "New_Year_Holidays",
                "2023-01-04": "New_Year_Holidays",
                "2023-01-05": "New_Year_Holidays",
                "2023-01-06": "New_Year_Holidays",
                "2023-01-07": "Christmas_Day",
                "2023-01-08": "New_Year_Holidays",
                "2023-02-23": "Defender_of_the_Fatherland_Day",
                "2023-03-08": "International_Womens_Day",
                "2023-05-01": "Spring_and_Labour_Holiday",
                "2023-05-09": "Victory_Day",
                "2023-06-12": "Day_of_Russia",
                "2023-11-04": "National_Unity_Day",
                "2023-02-24": "Additional_Holiday",
                "2023-05-08": "Additional_Holiday",
                "2023-11-06": "Additional_Holiday",
                # 2024
                "2024-01-01": "New_Year_Holidays",
                "2024-01-02": "New_Year_Holidays",
                "2024-01-03": "New_Year_Holidays",
                "2024-01-04": "New_Year_Holidays",
                "2024-01-05": "New_Year_Holidays",
                "2024-01-06": "New_Year_Holidays",
                "2024-01-07": "Christmas_Day",
                "2024-01-08": "New_Year_Holidays",
                "2024-02-23": "Defender_of_the_Fatherland_Day",
                "2024-03-08": "International_Womens_Day",
                "2024-05-01": "Spring_and_Labour_Holiday",
                "2024-05-09": "Victory_Day",
                "2024-06-12": "Day_of_Russia",
                "2024-11-04": "National_Unity_Day",
                "2024-04-29": "Additional_Holiday",
                "2024-04-30": "Additional_Holiday",
                "2024-05-10": "Additional_Holiday",
                "2024-12-30": "Additional_Holiday",
                "2024-12-31": "Additional_Holiday",
                # 2025
                "2025-01-01": "New_Year_Holidays",
                "2025-01-02": "New_Year_Holidays",
                "2025-01-03": "New_Year_Holidays",
                "2025-01-04": "New_Year_Holidays",
                "2025-01-05": "New_Year_Holidays",
                "2025-01-06": "New_Year_Holidays",
                "2025-01-07": "Christmas_Day",
                "2025-01-08": "New_Year_Holidays",
                "2025-02-23": "Defender_of_the_Fatherland_Day",
                "2025-03-08": "International_Womens_Day",
                "2025-05-01": "Spring_and_Labour_Holiday",
                "2025-05-09": "Victory_Day",
                "2025-06-12": "Day_of_Russia",
                "2025-11-04": "National_Unity_Day",
                "2025-05-02": "Additional_Holiday",
                "2025-05-08": "Additional_Holiday",
                "2025-06-13": "Additional_Holiday",
                "2025-11-03": "Additional_Holiday",
                "2025-12-31": "Additional_Holiday",
            }
            additional_working_days = {"2024-04-27", "2024-11-02", "2024-12-28", "2025-11-01"}
            return (
                {pd.to_datetime(k): v for k, v in holidays.items()},
                {pd.to_datetime(x) for x in additional_working_days},
            )

        holidays, additional_working_days = _holiday_calendar()

        df["season"] = df[month_col].map(
            {
                12: "Winter",
                1: "Winter",
                2: "Winter",
                3: "Spring",
                4: "Spring",
                5: "Spring",
                6: "Summer",
                7: "Summer",
                8: "Summer",
                9: "Autumn",
                10: "Autumn",
                11: "Autumn",
            }
        )
        df["is_winter"] = (df["season"] == "Winter").astype(int)
        df["is_spring"] = (df["season"] == "Spring").astype(int)
        df["is_autumn"] = (df["season"] == "Autumn").astype(int)
        df["is_quarter_start_month"] = df[month_col].isin([1, 4, 7, 10]).astype(int)
        df["is_quarter_end_month"] = df[month_col].isin([3, 6, 9, 12]).astype(int)
        df["is_halfyear_start_month"] = df[month_col].isin([1, 7]).astype(int)
        df["is_halfyear_end_month"] = df[month_col].isin([6, 12]).astype(int)
        df["is_year_start_month"] = (df[month_col] == 1).astype(int)
        df["is_year_end_month"] = (df[month_col] == 12).astype(int)
        df["month_in_halfyear"] = ((df[month_col] - 1) % 6 + 1).astype(int)
        df["month_name"] = df[month_col].map(
            {
                1: "January",
                2: "February",
                3: "March",
                4: "April",
                5: "May",
                6: "June",
                7: "July",
                8: "August",
                9: "September",
                10: "October",
                11: "November",
                12: "December",
            }
        )
        df["month_sin"] = np.sin(2 * np.pi * df[month_col] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df[month_col] / 12)
        df["month_sin_2"] = np.sin(2 * np.pi * 2 * df[month_col] / 12)
        df["month_cos_2"] = np.cos(2 * np.pi * 2 * df[month_col] / 12)
        df["prev_month"] = df[month_col].where(df[month_col] > 1, 13) - 1

        calendar = pd.DataFrame(
            {"date": pd.date_range(f"{min(years)}-01-01", f"{max(years)}-12-31", freq="D")}
        )
        calendar["year"] = calendar["date"].dt.year
        calendar["month"] = calendar["date"].dt.month
        calendar["weekday"] = calendar["date"].dt.dayofweek
        calendar["is_weekend"] = calendar["weekday"].isin([5, 6]).astype(int)
        calendar["holiday_name"] = calendar["date"].map(holidays)
        calendar["is_holiday"] = calendar["holiday_name"].notna().astype(int)
        calendar["is_additional_working_day"] = (
            calendar["date"].isin(additional_working_days).astype(int)
        )
        calendar["is_nonworking_day"] = (
            ((calendar["is_weekend"] == 1) | (calendar["is_holiday"] == 1))
            & (calendar["is_additional_working_day"] == 0)
        ).astype(int)
        calendar["is_working_day"] = 1 - calendar["is_nonworking_day"]
        calendar["is_may_holiday"] = (
            calendar["holiday_name"].isin(["Spring_and_Labour_Holiday", "Victory_Day"]).astype(int)
        )
        calendar["is_single_public_holiday"] = (
            calendar["holiday_name"]
            .isin(
                [
                    "Defender_of_the_Fatherland_Day",
                    "International_Womens_Day",
                    "Day_of_Russia",
                    "National_Unity_Day",
                ]
            )
            .astype(int)
        )
        calendar["is_additional_holiday"] = (
            calendar["holiday_name"] == "Additional_Holiday"
        ).astype(int)
        calendar["is_november_holiday"] = (calendar["holiday_name"] == "National_Unity_Day").astype(
            int
        )
        calendar["is_december_additional_holiday"] = (
            (calendar["month"] == 12) & (calendar["holiday_name"] == "Additional_Holiday")
        ).astype(int)

        month_calendar = calendar.groupby(["year", "month"], as_index=False).agg(
            n_days_in_month=("date", "count"),
            n_weekend_days=("is_weekend", "sum"),
            n_may_holiday_days=("is_may_holiday", "sum"),
            n_single_public_holiday_days=("is_single_public_holiday", "sum"),
            n_november_holiday_days=("is_november_holiday", "sum"),
            n_december_additional_holiday_days=("is_december_additional_holiday", "sum"),
            n_holiday_days=("is_holiday", "sum"),
            n_additional_holiday_days=("is_additional_holiday", "sum"),
            n_working_days=("is_working_day", "sum"),
            n_additional_working_days=("is_additional_working_day", "sum"),
        )
        month_calendar["is_leap_year"] = (
            (month_calendar["year"] % 4 == 0)
            & ((month_calendar["year"] % 100 != 0) | (month_calendar["year"] % 400 == 0))
        ).astype(int)
        month_calendar["holiday_month_type"] = month_calendar["month"].map(
            lambda m: (
                "new_year_holidays"
                if m == 1
                else "may_holidays"
                if m == 5
                else "single_public_holiday"
                if m in [2, 3, 6, 11]
                else "pre_new_year"
                if m == 12
                else "no_major_holiday"
            )
        )

        df = df.merge(
            month_calendar,
            left_on=[year_col, month_col],
            right_on=["year", "month"],
            how="left",
            suffixes=("", "_calendar"),
        ).drop(columns=["year_calendar", "month_calendar"], errors="ignore")

        df["is_middle_of_quarter"] = (((df[month_col] - 1) % 3 + 1) == 2).astype(int)
        df["is_pre_new_year_month"] = df[month_col].isin([11, 12]).astype(int)
        df["is_pre_may_holidays_month"] = (df[month_col] == 4).astype(int)
        df["is_back_to_school_month"] = df[month_col].isin([8, 9]).astype(int)
        df["is_business_activity_low_month"] = df[month_col].isin([1, 5, 7, 8]).astype(int)
        df["is_business_activity_high_month"] = df[month_col].isin([3, 4, 9, 10, 11]).astype(int)

        return df

    def _add_static(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Traffic
        df["total_traffic_per_hour"] = _col(df, "pedestrian_traffic_per_hour") + _col(
            df, "car_traffic_per_hour"
        )
        df["pedestrian_traffic_share"] = _safe_divide(
            _col(df, "pedestrian_traffic_per_hour"), df["total_traffic_per_hour"], fill=0.0
        )
        df["car_to_pedestrian_traffic_ratio"] = _safe_divide(
            _col(df, "car_traffic_per_hour"), _col(df, "pedestrian_traffic_per_hour"), fill=0.0
        )

        for c in [
            "population_size",
            "number_of_households",
            "pedestrian_traffic_per_hour",
            "car_traffic_per_hour",
        ]:
            if c in df.columns:
                df[f"log_{c}"] = np.log1p(df[c].clip(lower=0).fillna(0))

        if "settlement" in df.columns:
            df["is_moscow"] = (
                df["settlement"].astype(str).str.strip().str.lower().eq("москва г").astype(int)
            )
        else:
            df["is_moscow"] = 0

        df["is_million_plus_city"] = (_col(df, "population_size") >= 1_000_000).astype(int)

        def _city_type(pop, is_moscow):
            if is_moscow == 1:
                return "moscow"
            if pd.isna(pop) or pop <= 0:
                return "unknown_or_zero"
            if pop < 10_000:
                return "small_settlement"
            if pop < 50_000:
                return "small_town"
            if pop < 250_000:
                return "medium_city"
            if pop < 1_000_000:
                return "large_city"
            return "million_plus_city"

        df["population_city_type"] = [
            _city_type(p, m) for p, m in zip(_col(df, "population_size"), df["is_moscow"])
        ]

        for src, dst in [
            ("schools_300m", "has_school_nearby"),
            ("public_transport_stops_300m", "has_transport_nearby"),
            ("marketplaces_deliveries_parcel_lockers_100m", "has_marketplace_nearby"),
            ("medical_institutions_and_pharmacies_300m", "has_medical_nearby"),
            ("grocery_stores_500m", "has_grocery_competition"),
            ("pyaterochka_stores_500m", "has_pyaterochka_nearby"),
        ]:
            df[dst] = (_col(df, src) > 0).astype(int)

        df["is_high_grocery_competition"] = (_col(df, "grocery_stores_500m") >= 5).astype(int)
        df["is_high_pyaterochka_density"] = (_col(df, "pyaterochka_stores_500m") >= 3).astype(int)

        df["non_pyaterochka_grocery_stores_500m"] = (
            _col(df, "grocery_stores_500m") - _col(df, "pyaterochka_stores_500m")
        ).clip(lower=0)
        df["pyaterochka_share_among_grocery"] = _safe_divide(
            _col(df, "pyaterochka_stores_500m"), _col(df, "grocery_stores_500m"), fill=0.0
        )
        df["non_pyaterochka_competition_share"] = _safe_divide(
            df["non_pyaterochka_grocery_stores_500m"], _col(df, "grocery_stores_500m"), fill=0.0
        )
        df["grocery_per_1000_people"] = _safe_divide(
            _col(df, "grocery_stores_500m") * 1000, _col(df, "population_size"), fill=0.0
        )
        df["pyaterochka_per_1000_people"] = _safe_divide(
            _col(df, "pyaterochka_stores_500m") * 1000, _col(df, "population_size"), fill=0.0
        )
        df["competition_per_1000_households"] = _safe_divide(
            _col(df, "grocery_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0
        )
        df["pyaterochka_per_1000_households"] = _safe_divide(
            _col(df, "pyaterochka_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0
        )

        df["people_per_household"] = _safe_divide(
            _col(df, "population_size"), _col(df, "number_of_households"), fill=0.0
        )
        df["households_per_1000_people"] = _safe_divide(
            _col(df, "number_of_households") * 1000, _col(df, "population_size"), fill=0.0
        )

        df["traffic_per_1000_people"] = _safe_divide(
            df["total_traffic_per_hour"] * 1000, _col(df, "population_size"), fill=0.0
        )
        df["traffic_per_1000_households"] = _safe_divide(
            df["total_traffic_per_hour"] * 1000, _col(df, "number_of_households"), fill=0.0
        )
        df["pedestrian_traffic_per_1000_people"] = _safe_divide(
            _col(df, "pedestrian_traffic_per_hour") * 1000, _col(df, "population_size"), fill=0.0
        )

        df["social_infra_score"] = (
            _col(df, "medical_institutions_and_pharmacies_300m")
            + _col(df, "schools_300m")
            + _col(df, "public_transport_stops_300m")
        )
        df["commercial_infra_score"] = _col(
            df, "marketplaces_deliveries_parcel_lockers_100m"
        ) + _col(df, "grocery_stores_500m")
        df["poi_score"] = (
            _col(df, "marketplaces_deliveries_parcel_lockers_100m")
            + _col(df, "medical_institutions_and_pharmacies_300m")
            + _col(df, "schools_300m")
            + _col(df, "public_transport_stops_300m")
            + _col(df, "grocery_stores_500m")
        )
        df["poi_per_1000_households"] = _safe_divide(
            df["poi_score"] * 1000, _col(df, "number_of_households"), fill=0.0
        )
        df["public_transport_share_in_social_infra"] = _safe_divide(
            _col(df, "public_transport_stops_300m"), df["social_infra_score"], fill=0.0
        )
        df["medical_share_in_social_infra"] = _safe_divide(
            _col(df, "medical_institutions_and_pharmacies_300m"), df["social_infra_score"], fill=0.0
        )

        df["cash_register_hours_capacity"] = _col(df, "number_of_cash_registers") * _col(
            df, "working_hours_per_day"
        )
        df["traffic_per_cash_register"] = _safe_divide(
            df["total_traffic_per_hour"], _col(df, "number_of_cash_registers"), fill=0.0
        )
        df["traffic_per_working_hour"] = _safe_divide(
            df["total_traffic_per_hour"], _col(df, "working_hours_per_day"), fill=0.0
        )
        df["households_per_cash_register"] = _safe_divide(
            _col(df, "number_of_households"), _col(df, "number_of_cash_registers"), fill=0.0
        )
        df["population_per_cash_register"] = _safe_divide(
            _col(df, "population_size"), _col(df, "number_of_cash_registers"), fill=0.0
        )
        df["cash_registers_per_total_traffic"] = _safe_divide(
            _col(df, "number_of_cash_registers"), df["total_traffic_per_hour"], fill=0.0
        )

        if include_current_receipt_features:
            if {"avg_promo_items_per_receipt", "avg_items_per_receipt"}.issubset(df.columns):
                df["promo_items_share"] = _safe_divide(
                    _col(df, "avg_promo_items_per_receipt"),
                    _col(df, "avg_items_per_receipt"),
                    fill=0.0,
                )
            if {"avg_items_per_receipt", "working_hours_per_day"}.issubset(df.columns):
                df["items_x_working_hours"] = _col(df, "avg_items_per_receipt") * _col(
                    df, "working_hours_per_day"
                )

        df["alcohol_x_capacity"] = (
            _col(df, "alcohol_license_flag") * df["cash_register_hours_capacity"]
        )
        df["alcohol_x_total_traffic"] = (
            _col(df, "alcohol_license_flag") * df["total_traffic_per_hour"]
        )
        if include_current_receipt_features and "avg_items_per_receipt" in df.columns:
            df["alcohol_x_items"] = _col(df, "alcohol_license_flag") * _col(
                df, "avg_items_per_receipt"
            )

        df["traffic_x_total_poi"] = df["total_traffic_per_hour"] * df["poi_score"]
        df["traffic_x_grocery_competition"] = df["total_traffic_per_hour"] * _col(
            df, "grocery_stores_500m"
        )
        df["traffic_x_pyaterochka_competition"] = df["total_traffic_per_hour"] * _col(
            df, "pyaterochka_stores_500m"
        )
        df["pedestrian_traffic_x_transport"] = _col(df, "pedestrian_traffic_per_hour") * _col(
            df, "public_transport_stops_300m"
        )
        if "is_back_to_school_month" in df.columns:
            df["schools_x_back_to_school"] = _col(df, "schools_300m") * _col(
                df, "is_back_to_school_month"
            )

        # Store-level counts / exogenous group means. They use only exogenous known columns, not target.
        if id_col in df.columns:
            static_cols = [
                id_col,
                "settlement",
                "region",
                "opening_date_category",
                "trading_area_category",
                "total_traffic_per_hour",
                "grocery_stores_500m",
                "pyaterochka_stores_500m",
                "number_of_cash_registers",
                "poi_score",
            ]
            static_cols = [c for c in static_cols if c in df.columns]
            store_static = df[static_cols].drop_duplicates(subset=[id_col]).copy()

            for group_cols, new_name in [
                (["settlement"], "settlement_store_count"),
                (["region"], "region_store_count"),
                (["region", "trading_area_category"], "region_trading_area_store_count"),
                (["region", "opening_date_category"], "region_opening_age_store_count"),
            ]:
                if all(c in store_static.columns for c in group_cols):
                    cnt = (
                        store_static.groupby(group_cols, dropna=False)[id_col]
                        .nunique()
                        .rename(new_name)
                        .reset_index()
                    )
                    df = df.merge(cnt, on=group_cols, how="left")

            exog_cols = [
                c
                for c in [
                    "total_traffic_per_hour",
                    "grocery_stores_500m",
                    "pyaterochka_stores_500m",
                    "number_of_cash_registers",
                    "poi_score",
                ]
                if c in store_static.columns
            ]

            if exog_cols and "region" in store_static.columns:
                agg = (
                    store_static.groupby("region", dropna=False)[exog_cols]
                    .mean()
                    .add_prefix("region_")
                    .add_suffix("_mean")
                    .reset_index()
                )
                df = df.merge(agg, on="region", how="left")
            if exog_cols and "settlement" in store_static.columns:
                agg = (
                    store_static.groupby("settlement", dropna=False)[exog_cols]
                    .mean()
                    .add_prefix("settlement_")
                    .add_suffix("_mean")
                    .reset_index()
                )
                df = df.merge(agg, on="settlement", how="left")

        return df

    def _prepare_base_parts(parts: dict[str, pd.DataFrame]) -> tuple[dict[str, pd.DataFrame], dict]:
        checked = {}
        for name, p in parts.items():
            p = _ensure_time_year_month(p, name)
            _validate_required(p, {id_col, time_col}, name)
            checked[name] = p

        combined = _part_concat(checked)
        combined = _drop_preexisting_generated_cols(combined)
        combined = _ensure_time_year_month(combined, "combined")

        if add_calendar_features:
            combined = _add_calendar(combined)
            combined = _ensure_time_year_month(combined, "combined_after_calendar")

        if add_static_features:
            combined = _add_static(combined)

        parts_out = _split_parts(combined, list(parts.keys()))
        return parts_out, {"base_shape": combined.shape}

    # ============================================================
    # History feature blocks
    # ============================================================

    def _combine_history_future(
        history_df: pd.DataFrame, future_df: pd.DataFrame, future_name: str
    ):
        history_df = history_df.copy()
        future_df = future_df.copy()
        history_df["__part__"] = "history"
        future_df["__part__"] = future_name
        history_df["__row_order__"] = np.arange(len(history_df))
        future_df["__row_order__"] = np.arange(len(future_df))
        combined = pd.concat([history_df, future_df], axis=0, ignore_index=True, sort=False)
        combined = _ensure_time_year_month(combined, "history_future_combined")
        return combined

    def _restore_pair(combined: pd.DataFrame, future_name: str):
        history_features = (
            combined[combined["__part__"] == "history"]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        future_features = (
            combined[combined["__part__"] == future_name]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        return history_features, future_features

    def _merge_lag_feature(combined, history, source_col, lag, feature_name, by_cols):
        if source_col not in history.columns:
            return combined, False
        lag_df = history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")
        combined = combined.merge(
            lag_df[by_cols + [time_col, feature_name]], on=by_cols + [time_col], how="left"
        )
        return combined, True

    def _add_target_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_target_lags")
        _validate_unique_panel(history, "history_for_target_lags")

        created = []
        for lag in target_lags:
            fname = f"{target_col}_lag_{lag}"
            combined, ok = _merge_lag_feature(combined, history, target_col, lag, fname, [id_col])
            if ok:
                created.append(fname)
                if target_history_scale in {"log", "both"}:
                    log_name = f"log_{fname}"
                    combined[log_name] = np.log1p(combined[fname].clip(lower=0))
                    created.append(log_name)

        # Rolling features are based only on past history values; future target is ignored even if present.
        if target_rolling_windows:
            tmp = combined[[id_col, time_col, "__part__", target_col]].copy()
            tmp["__target_for_roll__"] = np.where(
                tmp["__part__"].eq("history"), tmp[target_col], np.nan
            )
            tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
            shifted = tmp.groupby(id_col, sort=False)["__target_for_roll__"].shift(1)
            for window in target_rolling_windows:
                rg = shifted.groupby(tmp[id_col], sort=False)
                mean_col = f"{target_col}_roll_mean_{window}"
                std_col = f"{target_col}_roll_std_{window}"
                cv_col = f"{target_col}_roll_cv_{window}"
                tmp[mean_col] = (
                    rg.rolling(window, min_periods=target_rolling_min_periods)
                    .mean()
                    .reset_index(level=0, drop=True)
                )
                tmp[std_col] = (
                    rg.rolling(window, min_periods=target_rolling_min_periods)
                    .std()
                    .reset_index(level=0, drop=True)
                )
                tmp[cv_col] = _safe_divide(tmp[std_col], tmp[mean_col], fill=np.nan)
                created.extend([mean_col, std_col, cv_col])
            tmp = tmp.sort_index()
            for c in created:
                if c in tmp.columns and c not in combined.columns:
                    combined[c] = tmp[c]

        def lag_col(lag):
            return f"{target_col}_lag_{lag}"

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_diff_lag_{a}_{b}"
                combined[c] = combined[lag_col(a)] - combined[lag_col(b)]
                created.append(c)

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_ratio_lag_{a}_{b}"
                combined[c] = _safe_divide(combined[lag_col(a)], combined[lag_col(b)], fill=np.nan)
                created.append(c)

        for window in target_rolling_windows:
            roll_mean = f"{target_col}_roll_mean_{window}"
            if {lag_col(1), roll_mean}.issubset(combined.columns):
                c = f"{target_col}_lag_1_to_roll_mean_{window}"
                combined[c] = _safe_divide(combined[lag_col(1)], combined[roll_mean], fill=np.nan)
                created.append(c)

        if {lag_col(12), f"{target_col}_roll_mean_12"}.issubset(combined.columns):
            c = f"{target_col}_lag_12_to_roll_mean_12"
            combined[c] = _safe_divide(
                combined[lag_col(12)], combined[f"{target_col}_roll_mean_12"], fill=np.nan
            )
            created.append(c)

        if {lag_col(12), lag_col(13)}.issubset(combined.columns):
            c1 = f"{target_col}_seasonal_diff_12_13"
            c2 = f"{target_col}_seasonal_ratio_12_13"
            combined[c1] = combined[lag_col(12)] - combined[lag_col(13)]
            combined[c2] = _safe_divide(combined[lag_col(12)], combined[lag_col(13)], fill=np.nan)
            created.extend([c1, c2])

        if target_history_scale in {"log", "both"} and add_log_target_dynamics:
            for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
                la, lb = f"log_{lag_col(a)}", f"log_{lag_col(b)}"
                if {la, lb}.issubset(combined.columns):
                    c = f"log_{target_col}_diff_lag_{a}_{b}"
                    combined[c] = combined[la] - combined[lb]
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_category_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_category_lags")

        specs = group_specs
        if specs is None:
            specs = [
                "region",
                "settlement",
                ["region", "trading_area_category"],
                ["region", "opening_date_category"],
                ["settlement", "trading_area_category"],
            ]
        norm_specs = []
        for spec in specs:
            spec = [spec] if isinstance(spec, str) else list(spec)
            if all(c in combined.columns for c in spec):
                norm_specs.append(spec)
        if not norm_specs:
            return (*_restore_pair(combined, future_name), [])

        created = []
        internal_store_lag = None
        store_lag_1 = f"{target_col}_lag_1"
        if store_lag_1 not in combined.columns:
            internal_store_lag = f"__{target_col}_store_lag_1__"
            combined, _ = _merge_lag_feature(
                combined, history, target_col, 1, internal_store_lag, [id_col]
            )
            store_lag_1 = internal_store_lag

        for group_cols in norm_specs:
            prefix = "__".join(group_cols)
            agg_col = f"__{prefix}_{target_col}_{group_agg_stat}__"
            monthly_agg = (
                history.groupby(group_cols + [time_col], dropna=False)[target_col]
                .agg(group_agg_stat)
                .reset_index()
                .rename(columns={target_col: agg_col})
            )
            monthly_agg[time_col] = monthly_agg[time_col].astype(int) + int(group_lag)
            group_lag_col = f"{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
            monthly_agg = monthly_agg.rename(columns={agg_col: group_lag_col})
            combined = combined.merge(
                monthly_agg[group_cols + [time_col, group_lag_col]],
                on=group_cols + [time_col],
                how="left",
            )

            if add_group_lag_feature:
                created.append(group_lag_col)

            if add_group_relative_ratio:
                c = f"{target_col}_lag_1_to_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = _safe_divide(
                    combined[store_lag_1], combined[group_lag_col], fill=np.nan
                )
                created.append(c)

            if add_group_relative_diff:
                c = f"{target_col}_lag_1_minus_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = combined[store_lag_1] - combined[group_lag_col]
                created.append(c)

        if internal_store_lag is not None:
            combined = combined.drop(columns=[internal_store_lag], errors="ignore")

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_monthly_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col}, "history_for_monthly_lags")
        _validate_unique_panel(history, "history_for_monthly_lags")

        existing_monthly = [c for c in current_monthly_cols if c in history.columns]
        if not existing_monthly:
            return (*_restore_pair(combined, future_name), [])

        if monthly_feature_config is None:
            cfg_all = {
                "avg_promo_items_per_receipt": {
                    "lags": (1, 3, 12),
                    "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"),
                    "diff_pairs": ((1, 12),),
                    "ratio_pairs": ((1, 12),),
                },
                "avg_items_per_receipt": {
                    "lags": (1, 3, 12),
                    "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"),
                    "diff_pairs": ((1, 12),),
                    "ratio_pairs": ((1, 12),),
                },
                "avg_cancellations": {
                    "lags": (1, 3, 12),
                    "rolling_windows": (3, 12),
                    "rolling_stats": ("mean", "std", "cv"),
                    "diff_pairs": ((1, 12),),
                    "ratio_pairs": ((1, 12),),
                },
                "working_hours_per_day": {
                    "lags": (1,),
                    "rolling_windows": (),
                    "rolling_stats": (),
                    "diff_pairs": (),
                    "ratio_pairs": (),
                },
            }
        else:
            cfg_all = monthly_feature_config

        created = []

        for colname in existing_monthly:
            cfg = cfg_all.get(
                colname,
                {
                    "lags": (1, 3, 12),
                    "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"),
                    "diff_pairs": ((1, 12),),
                    "ratio_pairs": ((1, 12),),
                },
            )

            # Time-aware lags.
            for lag in tuple(cfg.get("lags", ())):
                c = f"{colname}_lag_{lag}"
                combined, ok = _merge_lag_feature(combined, history, colname, lag, c, [id_col])
                if ok:
                    created.append(c)
                    if monthly_history_scale in {"log", "both"}:
                        lc = f"log_{colname}_lag_{lag}"
                        combined[lc] = np.log1p(combined[c].clip(lower=0))
                        created.append(lc)

            # Rolling over past observed history.
            windows = tuple(cfg.get("rolling_windows", ()))
            stats = tuple(cfg.get("rolling_stats", ()))
            if windows and stats:
                tmp = combined[[id_col, time_col, "__part__", colname]].copy()
                tmp["__x__"] = np.where(tmp["__part__"].eq("history"), tmp[colname], np.nan)
                tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
                shifted = tmp.groupby(id_col, sort=False)["__x__"].shift(1)
                for window in windows:
                    rg = shifted.groupby(tmp[id_col], sort=False)
                    mean_s = std_s = min_s = max_s = None
                    if "mean" in stats or "cv" in stats:
                        mean_s = (
                            rg.rolling(window, min_periods=monthly_rolling_min_periods)
                            .mean()
                            .reset_index(level=0, drop=True)
                        )
                        if "mean" in stats:
                            c = f"{colname}_roll_mean_{window}"
                            tmp[c] = mean_s
                            created.append(c)
                    if "std" in stats or "cv" in stats:
                        std_s = (
                            rg.rolling(window, min_periods=monthly_rolling_min_periods)
                            .std()
                            .reset_index(level=0, drop=True)
                        )
                        if "std" in stats:
                            c = f"{colname}_roll_std_{window}"
                            tmp[c] = std_s
                            created.append(c)
                    if "min" in stats or "range" in stats:
                        min_s = (
                            rg.rolling(window, min_periods=monthly_rolling_min_periods)
                            .min()
                            .reset_index(level=0, drop=True)
                        )
                        if "min" in stats:
                            c = f"{colname}_roll_min_{window}"
                            tmp[c] = min_s
                            created.append(c)
                    if "max" in stats or "range" in stats:
                        max_s = (
                            rg.rolling(window, min_periods=monthly_rolling_min_periods)
                            .max()
                            .reset_index(level=0, drop=True)
                        )
                        if "max" in stats:
                            c = f"{colname}_roll_max_{window}"
                            tmp[c] = max_s
                            created.append(c)
                    if "range" in stats:
                        c = f"{colname}_roll_range_{window}"
                        tmp[c] = max_s - min_s
                        created.append(c)
                    if "cv" in stats:
                        c = f"{colname}_roll_cv_{window}"
                        tmp[c] = _safe_divide(std_s, mean_s, fill=np.nan)
                        created.append(c)
                tmp = tmp.sort_index()
                for c in created:
                    if c in tmp.columns and c not in combined.columns:
                        combined[c] = tmp[c]

            for a, b in tuple(cfg.get("diff_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_diff_lag_{a}_{b}"
                    combined[c] = combined[ca] - combined[cb]
                    created.append(c)

            for a, b in tuple(cfg.get("ratio_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_ratio_lag_{a}_{b}"
                    combined[c] = _safe_divide(combined[ca], combined[cb], fill=np.nan)
                    created.append(c)

        if add_monthly_cross_lags:
            for lag in monthly_cross_lags:
                promo = f"avg_promo_items_per_receipt_lag_{lag}"
                items = f"avg_items_per_receipt_lag_{lag}"
                canc = f"avg_cancellations_lag_{lag}"
                hours = f"working_hours_per_day_lag_{lag}"

                if "promo_share" in monthly_cross_feature_types and {promo, items}.issubset(
                    combined.columns
                ):
                    c = f"promo_items_share_lag_{lag}"
                    combined[c] = _safe_divide(combined[promo], combined[items], fill=np.nan)
                    created.append(c)
                if "promo_x_items" in monthly_cross_feature_types and {promo, items}.issubset(
                    combined.columns
                ):
                    c = f"promo_x_items_lag_{lag}"
                    combined[c] = combined[promo] * combined[items]
                    created.append(c)
                if "items_x_hours" in monthly_cross_feature_types and {items, hours}.issubset(
                    combined.columns
                ):
                    c = f"items_x_working_hours_lag_{lag}"
                    combined[c] = combined[items] * combined[hours]
                    created.append(c)
                if "cancellations_per_item" in monthly_cross_feature_types and {
                    canc,
                    items,
                }.issubset(combined.columns):
                    c = f"cancellations_per_item_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[items], fill=np.nan)
                    created.append(c)
                if "cancellations_per_hour" in monthly_cross_feature_types and {
                    canc,
                    hours,
                }.issubset(combined.columns):
                    c = f"cancellations_per_working_hour_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[hours], fill=np.nan)
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)

        if drop_current_monthly_cols:
            combined = combined.drop(columns=list(existing_monthly), errors="ignore")
        if drop_current_monthly_derived_cols:
            combined = combined.drop(columns=list(current_monthly_derived_cols), errors="ignore")

        return (*_restore_pair(combined, future_name), created)

    # ============================================================
    # Final cleanup
    # ============================================================

    def _future_unknown_cols():
        cols = []
        if drop_current_monthly_cols:
            cols.extend(list(current_monthly_cols))
        if drop_current_monthly_derived_cols:
            cols.extend(list(current_monthly_derived_cols))
        if not working_hours_future_known:
            cols.extend(
                [
                    "working_hours_per_day",
                    "cash_register_hours_capacity",
                    "traffic_per_working_hour",
                    "alcohol_x_capacity",
                    "items_x_working_hours",
                ]
            )
        if drop_year_col:
            cols.append(year_col)
        return sorted(set(cols))

    def _obvious_duplicate_cols():
        cols = [
            f"{target_col}_roll_mean_3",
            f"{target_col}_roll_mean_12",
            f"{target_col}_lag_13",
            f"{target_col}_diff_lag_3_6",
            f"{target_col}_ratio_lag_3_6",
            "avg_items_per_receipt_diff_lag_1_12",
            "avg_cancellations_roll_mean_3",
            "avg_cancellations_roll_mean_12",
            "cancellations_per_item_lag_1",
            "cancellations_per_item_lag_3",
            "cancellations_per_item_lag_12",
            "settlement_store_count",
            "total_traffic_per_hour",
            "non_pyaterochka_grocery_stores_500m",
        ]

        if target_history_scale == "raw":
            cols.extend([f"log_{target_col}_lag_{l}" for l in target_lags])
            cols.extend([c for c in []])
        elif target_history_scale == "log":
            raw_lags = [f"{target_col}_lag_{l}" for l in target_lags if l != 1]
            if not keep_raw_pto_lag_1_anchor_for_log:
                raw_lags.append(f"{target_col}_lag_1")
            cols.extend(raw_lags)
            cols.extend(
                [
                    f"{target_col}_diff_lag_1_2",
                    f"{target_col}_diff_lag_1_3",
                    f"{target_col}_diff_lag_1_6",
                    f"{target_col}_diff_lag_1_12",
                    f"{target_col}_diff_lag_3_6",
                    f"{target_col}_ratio_lag_1_2",
                    f"{target_col}_ratio_lag_1_3",
                    f"{target_col}_ratio_lag_1_6",
                    f"{target_col}_ratio_lag_1_12",
                    f"{target_col}_ratio_lag_3_6",
                    f"{target_col}_ratio_lag_6_12",
                    f"{target_col}_seasonal_diff_12_13",
                    f"{target_col}_seasonal_ratio_12_13",
                ]
            )

        if monthly_history_scale == "raw":
            for c in current_monthly_cols:
                cols.extend([f"log_{c}_lag_1", f"log_{c}_lag_3", f"log_{c}_lag_12"])
        elif monthly_history_scale == "log":
            for c in current_monthly_cols:
                cols.extend([f"{c}_lag_3", f"{c}_lag_12"])

        if extra_drop_cols is not None:
            cols.extend(list(extra_drop_cols))
        return sorted(set(cols))

    def _apply_final_clean(train_part, future_part):
        train_part = train_part.copy()
        future_part = future_part.copy()
        drop_cols = _future_unknown_cols()
        if drop_obvious_duplicates:
            drop_cols.extend(_obvious_duplicate_cols())
        drop_cols = sorted(
            set(c for c in drop_cols if c in train_part.columns or c in future_part.columns)
        )
        train_part = train_part.drop(columns=drop_cols, errors="ignore")
        future_part = future_part.drop(columns=drop_cols, errors="ignore")

        # Replace infinities only; do not fill all lag NaNs unless requested.
        for df_ in (train_part, future_part):
            num_cols = df_.select_dtypes(include=[np.number]).columns
            df_[num_cols] = df_[num_cols].replace([np.inf, -np.inf], np.nan)
            if fill_history_na:
                df_[num_cols] = df_[num_cols].fillna(fill_value)

        # Align columns in stable order.
        ordered_cols = list(train_part.columns)
        for c in future_part.columns:
            if c not in ordered_cols:
                ordered_cols.append(c)
        for c in ordered_cols:
            if c not in train_part.columns:
                train_part[c] = np.nan
            if c not in future_part.columns:
                future_part[c] = np.nan
        train_part = train_part[ordered_cols]
        future_part = future_part[ordered_cols]
        return train_part, future_part, drop_cols

    def _downcast(df):
        if not downcast_float_features:
            return df
        df = df.copy()
        float_cols = [c for c in df.select_dtypes(include=["float64"]).columns if c != target_col]
        if float_cols:
            df[float_cols] = df[float_cols].astype("float32")
        return df

    def _generate_pair(history_base, future_base, future_name):
        info = {}
        h, f = history_base.copy(), future_base.copy()

        if add_target_history:
            h, f, created = _add_target_history_pair(h, f, future_name)
            info["target_history_features"] = created
        else:
            info["target_history_features"] = []

        if add_category_target_history:
            h, f, created = _add_category_history_pair(h, f, future_name)
            info["category_target_history_features"] = created
        else:
            info["category_target_history_features"] = []

        if add_non_target_monthly_history:
            h, f, created = _add_monthly_history_pair(h, f, future_name)
            info["non_target_monthly_history_features"] = created
        else:
            info["non_target_monthly_history_features"] = []
            if drop_current_monthly_cols:
                h = h.drop(
                    columns=[c for c in current_monthly_cols if c in h.columns], errors="ignore"
                )
                f = f.drop(
                    columns=[c for c in current_monthly_cols if c in f.columns], errors="ignore"
                )
            if drop_current_monthly_derived_cols:
                h = h.drop(
                    columns=[c for c in current_monthly_derived_cols if c in h.columns],
                    errors="ignore",
                )
                f = f.drop(
                    columns=[c for c in current_monthly_derived_cols if c in f.columns],
                    errors="ignore",
                )

        h, f, dropped = _apply_final_clean(h, f)
        info["dropped_final_cols"] = dropped
        h = _downcast(h)
        f = _downcast(f)
        return h, f, info

    # ============================================================
    # Main flow
    # ============================================================

    train_df = _ensure_time_year_month(train_df, "train_df")
    _validate_required(train_df, {id_col, time_col, target_col}, "train_df")
    _validate_unique_panel(train_df, "train_df")

    if make_val_from_train:
        if val_df is not None:
            raise ValueError("Нельзя одновременно передать val_df и make_val_from_train=True.")
        if val_time_idx is None:
            val_time_idx = int(train_df[time_col].max())
        val_df = train_df[train_df[time_col] == val_time_idx].copy()
        train_df = train_df[train_df[time_col] < val_time_idx].copy()
        if verbose:
            print(f"Auto val_time_idx: {val_time_idx}")
            print("Auto train shape:", train_df.shape)
            print("Auto val shape:", val_df.shape)

    if val_df is not None:
        val_df = _ensure_time_year_month(val_df, "val_df")
        _validate_required(val_df, {id_col, time_col}, "val_df")
        _validate_unique_panel(val_df, "val_df")

    if test_df is not None:
        test_df = _ensure_time_year_month(test_df, "test_df")
        _validate_required(test_df, {id_col, time_col}, "test_df")
        _validate_unique_panel(test_df, "test_df")
        if target_col not in test_df.columns:
            test_df[target_col] = np.nan

    output = {
        "train_features": None,
        "val_features": None,
        "test_train_features": None,
        "test_features": None,
        "feature_cols": None,
        "categorical_cols": None,
        "feature_info": {},
    }

    if val_df is not None:
        base_parts, base_info = _prepare_base_parts({"train": train_df, "val": val_df})
        train_features, val_features, val_info = _generate_pair(
            base_parts["train"], base_parts["val"], "val"
        )
        output["train_features"] = train_features
        output["val_features"] = val_features
        output["feature_info"]["val"] = {**base_info, **val_info}
        if verbose:
            print("Generated train_features:", train_features.shape)
            print("Generated val_features:", val_features.shape)

    if test_df is not None:
        if val_df is not None and use_val_as_test_history:
            test_history = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
        else:
            test_history = train_df.copy()
        test_history = _ensure_time_year_month(test_history, "test_history")
        _validate_unique_panel(test_history, "test_history")

        base_parts, base_info = _prepare_base_parts({"train": test_history, "test": test_df})
        test_train_features, test_features, test_info = _generate_pair(
            base_parts["train"], base_parts["test"], "test"
        )
        output["test_train_features"] = test_train_features
        output["test_features"] = test_features
        output["feature_info"]["test"] = {**base_info, **test_info}
        if verbose:
            print("Generated test_train_features:", test_train_features.shape)
            print("Generated test_features:", test_features.shape)

    # Select columns for model.
    ref = (
        output["test_train_features"]
        if output["test_train_features"] is not None
        else output["train_features"]
    )
    if ref is not None:
        exclude = {target_col}
        feature_cols = [c for c in ref.columns if c not in exclude]
        categorical_cols = [
            c
            for c in feature_cols
            if c in ref.columns
            and (pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c]))
        ]
        output["feature_cols"] = feature_cols
        output["categorical_cols"] = categorical_cols

    gc.collect()
    return output


def _ensure_time_year_month_external(
    df: pd.DataFrame,
    *,
    year_col: str,
    month_col: str,
    time_col: str,
    min_year: int,
    name: str = "df",
) -> pd.DataFrame:
    """Внешний helper для wrapper/enhancer: гарантирует year/month/time."""
    out = df.copy()
    has_year_month = year_col in out.columns and month_col in out.columns
    has_time = time_col in out.columns

    if not has_year_month and not has_time:
        raise ValueError(f"В {name} нет ни year/month, ни time_col='{time_col}'.")

    if has_year_month:
        out[year_col] = pd.to_numeric(out[year_col], errors="raise").astype(int)
        out[month_col] = pd.to_numeric(out[month_col], errors="raise").astype(int)
        out[time_col] = ((out[year_col] - int(min_year)) * 12 + (out[month_col] - 1)).astype(int)
    else:
        out[time_col] = pd.to_numeric(out[time_col], errors="raise").astype(int)
        out[year_col] = (int(min_year) + out[time_col] // 12).astype(int)
        out[month_col] = (out[time_col] % 12 + 1).astype(int)

    return out


def _calendar_month_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (time_s.astype(int) % 12 + 1).astype(int)


def _year_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (int(min_year) + time_s.astype(int) // 12).astype(int)


def _days_in_month_from_year_month(year_s: pd.Series, month_s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(
        {
            "year": pd.to_numeric(year_s, errors="coerce").astype("Int64"),
            "month": pd.to_numeric(month_s, errors="coerce").astype("Int64"),
            "day": 1,
        },
        errors="coerce",
    )
    return dt.dt.days_in_month.astype("float32")


def _safe_divide_external(a, b, fill=np.nan):
    a = a if isinstance(a, pd.Series) else pd.Series(a)
    b = b if isinstance(b, pd.Series) else pd.Series(b)
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    out = out.replace([np.inf, -np.inf], np.nan)
    if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
        out = out.fillna(fill)
    return out


def _safe_log_external(x, eps: float = 1e-9):
    return np.log(np.clip(pd.Series(x).astype(float), eps, None))


def _assign_lag1_rto_size_bin(
    df: pd.DataFrame,
    *,
    time_col: str,
    lag1_col: str,
    out_col: str = "lag1_rto_size_bin",
    n_bins: int = 5,
) -> pd.Series:
    """
    Cross-sectional size bin by pto_lag_1 inside each forecast month.
    Uses only lagged target values, therefore it is safe for future rows.
    """
    result = pd.Series(np.nan, index=df.index, dtype="object")

    if lag1_col not in df.columns:
        return result

    for _, idx in df.groupby(time_col, sort=False).groups.items():
        s = pd.to_numeric(df.loc[idx, lag1_col], errors="coerce")
        valid = s.notna()
        if valid.sum() == 0:
            continue
        if valid.sum() == 1 or s[valid].nunique(dropna=True) == 1:
            result.loc[s[valid].index] = "q3"
            continue
        q = int(min(n_bins, valid.sum(), s[valid].nunique(dropna=True)))
        if q < 2:
            result.loc[s[valid].index] = "q3"
            continue
        try:
            codes = pd.qcut(
                s[valid].rank(method="first"),
                q=q,
                labels=False,
                duplicates="drop",
            )
            result.loc[s[valid].index] = [
                f"q{int(x) + 1}" if pd.notna(x) else np.nan for x in codes
            ]
        except ValueError:
            result.loc[s[valid].index] = "q3"

    return result


def _expanding_median_factor_from_history(
    combined: pd.DataFrame,
    observed: pd.DataFrame,
    *,
    group_cols: list[str],
    time_col: str,
    obs_col: str,
    out_col: str,
) -> pd.Series:
    """
    For every row in combined returns median(obs_col) over historical observations
    with the same group key and strictly earlier time_col.

    Implementation detail: query rows are sorted before observation rows at the
    same time index, so same-month target is never used for the same row/month.
    """
    keys = list(group_cols) + [time_col]
    result = pd.Series(np.nan, index=combined.index, dtype="float64")

    if not group_cols or obs_col not in observed.columns:
        return result
    if any(c not in combined.columns for c in keys) or any(c not in observed.columns for c in keys):
        return result

    obs = observed[keys + [obs_col]].dropna(subset=[obs_col]).copy()
    if obs.empty:
        return result

    obs_gt = (
        obs.groupby(keys, dropna=False, as_index=False)[obs_col]
        .median()
        .rename(columns={obs_col: "__obs_factor__"})
    )
    if obs_gt.empty:
        return result

    queries = combined[keys].copy()
    queries["__query_index__"] = combined.index
    queries["__obs_factor__"] = np.nan
    queries["__is_query__"] = 1
    queries["__order__"] = 0

    obs_gt["__query_index__"] = -1
    obs_gt["__is_query__"] = 0
    obs_gt["__order__"] = 1

    stack = pd.concat([queries, obs_gt], ignore_index=True, sort=False)
    stack = stack.sort_values(group_cols + [time_col, "__order__"], kind="mergesort")

    stack[out_col] = stack.groupby(group_cols, dropna=False)["__obs_factor__"].transform(
        lambda s: s.expanding(min_periods=1).median()
    )

    q = stack[stack["__is_query__"].eq(1)][["__query_index__", out_col]].copy()
    q = q.drop_duplicates("__query_index__", keep="last")
    result.loc[q["__query_index__"].astype(int).values] = q[out_col].values
    return result


def _add_expanding_calendar_heuristic_features(
    combined: pd.DataFrame,
    *,
    id_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
    eps: float = 1e-9,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Adds the old expanding calendar heuristic block without target leakage.

    Exposed features:
    - lag1_rto_size_bin;
    - same_month_calendar_factor;
    - same_month_region_calendar_factor;
    - same_month_population_type_calendar_factor;
    - same_month_lag1_size_calendar_factor;
    - calendar_growth_multiplier;
    - calendar_heuristic_*_prediction;
    - log_calendar_heuristic_prediction;
    - log_pto_lag_1_to_calendar_heuristic;
    - log_pto_lag_1_to_day_scaled_lag1.

    Internal observed_calendar_factor is deliberately not returned as a model
    feature, because it uses current target on history rows.
    """
    out = combined.copy()
    created: list[str] = []

    def mark(col: str):
        if col in out.columns and col not in created:
            created.append(col)

    pto_lag_1 = f"{target_col}_lag_1"
    required = {
        "__part__",
        time_col,
        target_col,
        pto_lag_1,
        "day_scaled_lag1_prediction",
        "month_day_ratio_to_previous_month",
    }
    if not required.issubset(out.columns):
        return out, created

    if "calendar_month_num" not in out.columns:
        out["calendar_month_num"] = _calendar_month_from_time(out[time_col], min_year)
        mark("calendar_month_num")

    out["lag1_rto_size_bin"] = _assign_lag1_rto_size_bin(
        out,
        time_col=time_col,
        lag1_col=pto_lag_1,
        out_col="lag1_rto_size_bin",
        n_bins=5,
    )
    mark("lag1_rto_size_bin")

    # Useful log-ratio against the pure day-scaled anchor.
    if f"log_{target_col}_lag_1" not in out.columns:
        out[f"log_{target_col}_lag_1"] = _safe_log_external(out[pto_lag_1], eps=eps)
        mark(f"log_{target_col}_lag_1")
    if "log_day_scaled_lag1_prediction" not in out.columns:
        out["log_day_scaled_lag1_prediction"] = _safe_log_external(
            out["day_scaled_lag1_prediction"], eps=eps
        )
        mark("log_day_scaled_lag1_prediction")

    out["log_pto_lag_1_to_day_scaled_lag1"] = (
        out[f"log_{target_col}_lag_1"] - out["log_day_scaled_lag1_prediction"]
    )
    mark("log_pto_lag_1_to_day_scaled_lag1")

    hist_mask = out["__part__"].eq("history") & out[target_col].notna()
    observed = out.loc[hist_mask].copy()
    observed["__observed_calendar_factor__"] = _safe_divide_external(
        observed[target_col],
        observed["day_scaled_lag1_prediction"],
        fill=np.nan,
    )
    observed["__observed_calendar_factor__"] = observed["__observed_calendar_factor__"].replace(
        [np.inf, -np.inf], np.nan
    )
    observed = observed.dropna(subset=["__observed_calendar_factor__"])

    if observed.empty:
        return out, created

    factor_specs = [
        (["calendar_month_num"], "same_month_calendar_factor", None),
    ]
    if "region" in out.columns:
        factor_specs.append(
            (
                ["calendar_month_num", "region"],
                "same_month_region_calendar_factor",
                "same_month_calendar_factor",
            )
        )
    if "population_city_type" in out.columns:
        factor_specs.append(
            (
                ["calendar_month_num", "population_city_type"],
                "same_month_population_type_calendar_factor",
                "same_month_calendar_factor",
            )
        )
    if "lag1_rto_size_bin" in out.columns:
        factor_specs.append(
            (
                ["calendar_month_num", "lag1_rto_size_bin"],
                "same_month_lag1_size_calendar_factor",
                "same_month_calendar_factor",
            )
        )

    for group_cols, factor_col, fallback_col in factor_specs:
        if any(c not in out.columns for c in group_cols):
            continue
        factor = _expanding_median_factor_from_history(
            out,
            observed,
            group_cols=group_cols,
            time_col=time_col,
            obs_col="__observed_calendar_factor__",
            out_col=factor_col,
        )
        out[factor_col] = factor
        if fallback_col is not None and fallback_col in out.columns:
            out[factor_col] = out[factor_col].fillna(out[fallback_col])
        mark(factor_col)

    if "same_month_calendar_factor" in out.columns:
        out["calendar_growth_multiplier"] = (
            out["month_day_ratio_to_previous_month"] * out["same_month_calendar_factor"]
        )
        out["calendar_heuristic_prediction"] = out[pto_lag_1] * out["calendar_growth_multiplier"]
        out["log_calendar_heuristic_prediction"] = _safe_log_external(
            out["calendar_heuristic_prediction"], eps=eps
        )
        out["log_pto_lag_1_to_calendar_heuristic"] = (
            out[f"log_{target_col}_lag_1"] - out["log_calendar_heuristic_prediction"]
        )
        mark("calendar_growth_multiplier")
        mark("calendar_heuristic_prediction")
        mark("log_calendar_heuristic_prediction")
        mark("log_pto_lag_1_to_calendar_heuristic")

    pred_specs = [
        ("same_month_region_calendar_factor", "calendar_heuristic_region_prediction"),
        (
            "same_month_population_type_calendar_factor",
            "calendar_heuristic_population_type_prediction",
        ),
        ("same_month_lag1_size_calendar_factor", "calendar_heuristic_lag1_size_prediction"),
    ]
    for factor_col, pred_col in pred_specs:
        if factor_col not in out.columns:
            continue
        out[pred_col] = out[pto_lag_1] * out["month_day_ratio_to_previous_month"] * out[factor_col]
        out[f"log_{pred_col}"] = _safe_log_external(out[pred_col], eps=eps)
        mark(pred_col)
        mark(f"log_{pred_col}")

    created = sorted(set(c for c in created if c in out.columns))
    out[created] = out[created].replace([np.inf, -np.inf], np.nan)
    return out, created


def _add_integrated_anchor_features_pair(
    history_features: pd.DataFrame,
    future_features: pd.DataFrame,
    history_raw: pd.DataFrame,
    future_raw: pd.DataFrame,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Добавляет идеи из build_lag1_features_for_pipeline,
    build_anchor_features_for_pipeline и build_march_hierarchy_ops_features_for_pipeline
    к уже собранным leak-free features.

    Важная логика:
    - лаги target и monthly-cols строятся time-aware merge'ем из history_raw;
    - current target future_features не используется;
    - мартовские коэффициенты 2024 считаются только по history_raw;
    - expanding calendar heuristic считается только по history rows и строго прошлым time_idx;
    - мартовские коэффициенты доступны только для строк после марта 2024, чтобы не
      протаскивать target марта 2024 в более ранние train-строки.
    """
    h_feat = history_features.copy()
    f_feat = future_features.copy()

    h_feat["__part__"] = "history"
    f_feat["__part__"] = "future"
    h_feat["__row_order__"] = np.arange(len(h_feat))
    f_feat["__row_order__"] = np.arange(len(f_feat))

    combined = pd.concat([h_feat, f_feat], ignore_index=True, sort=False)
    combined = _ensure_time_year_month_external(
        combined,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="combined_features",
    )

    history_raw = _ensure_time_year_month_external(
        history_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="history_raw",
    )
    future_raw = _ensure_time_year_month_external(
        future_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="future_raw",
    )

    if id_col not in combined.columns:
        raise ValueError(f"В features нет id_col='{id_col}'.")
    if target_col not in history_raw.columns:
        raise ValueError(f"В history_raw нет target_col='{target_col}'.")

    created: list[str] = []

    def _mark(col):
        if col in combined.columns and col not in created:
            created.append(col)

    def _merge_history_lag(source_history, source_col, lag, feature_name, by_cols):
        nonlocal combined
        if source_col not in source_history.columns:
            return False

        lag_df = source_history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")

        combined = combined.drop(columns=[feature_name], errors="ignore")
        combined = combined.merge(
            lag_df[by_cols + [time_col, feature_name]],
            on=by_cols + [time_col],
            how="left",
        )
        _mark(feature_name)
        return True

    # ------------------------------------------------------------
    # 1. Calendar day features + target lags/daily lags
    # ------------------------------------------------------------
    combined["calendar_month_num"] = _calendar_month_from_time(combined[time_col], min_year)
    combined["calendar_year"] = _year_from_time(combined[time_col], min_year)
    combined["days_in_month"] = _days_in_month_from_year_month(
        combined["calendar_year"], combined["calendar_month_num"]
    )
    _mark("calendar_month_num")
    _mark("days_in_month")

    for lag in target_lags:
        lag = int(lag)
        pto_lag_col = f"{target_col}_lag_{lag}"
        _merge_history_lag(history_raw, target_col, lag, pto_lag_col, [id_col])

        lag_time = combined[time_col].astype(int) - lag
        lag_year = _year_from_time(lag_time, min_year)
        lag_month = _calendar_month_from_time(lag_time, min_year)
        days_lag_col = f"days_lag_{lag}"
        combined[days_lag_col] = _days_in_month_from_year_month(lag_year, lag_month)
        _mark(days_lag_col)

        daily_col = f"daily_{target_col}_lag_{lag}"
        combined[daily_col] = _safe_divide_external(
            combined[pto_lag_col],
            combined[days_lag_col],
            fill=np.nan,
        )
        _mark(daily_col)

        log_pto_col = f"log_{target_col}_lag_{lag}"
        log_daily_col = f"log_daily_{target_col}_lag_{lag}"
        combined[log_pto_col] = _safe_log_external(combined[pto_lag_col], eps=eps)
        combined[log_daily_col] = _safe_log_external(combined[daily_col], eps=eps)
        _mark(log_pto_col)
        _mark(log_daily_col)

    if "days_lag_1" in combined.columns:
        combined["days_in_prev_month"] = combined["days_lag_1"]
        _mark("days_in_prev_month")

    if add_calendar_anchor_features and {
        f"{target_col}_lag_1",
        "days_in_month",
        "days_in_prev_month",
    }.issubset(combined.columns):
        combined["month_day_ratio_to_previous_month"] = _safe_divide_external(
            combined["days_in_month"],
            combined["days_in_prev_month"],
            fill=np.nan,
        )
        combined["day_scaled_lag1_prediction"] = (
            combined[f"{target_col}_lag_1"] * combined["month_day_ratio_to_previous_month"]
        )
        combined["log_day_scaled_lag1_prediction"] = _safe_log_external(
            combined["day_scaled_lag1_prediction"], eps=eps
        )
        _mark("month_day_ratio_to_previous_month")
        _mark("day_scaled_lag1_prediction")
        _mark("log_day_scaled_lag1_prediction")

        for lag in [2, 3, 6, 12]:
            daily_col = f"daily_{target_col}_lag_{lag}"
            pred_col = f"day_scaled_lag{lag}_prediction"
            if daily_col in combined.columns:
                combined[pred_col] = combined[daily_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

    if add_calendar_anchor_features and add_expanding_calendar_heuristic_features:
        combined, calendar_created = _add_expanding_calendar_heuristic_features(
            combined,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            eps=eps,
        )
        for c in calendar_created:
            _mark(c)

    # ------------------------------------------------------------
    # 2. Recent daily baselines + trend anchors + same-transition anchors
    # ------------------------------------------------------------
    if add_daily_anchor_features:
        daily_1 = f"daily_{target_col}_lag_1"
        daily_2 = f"daily_{target_col}_lag_2"
        daily_3 = f"daily_{target_col}_lag_3"

        if {daily_1, daily_2}.issubset(combined.columns):
            combined["recent_daily_mean_2"] = combined[[daily_1, daily_2]].mean(axis=1)
            combined["recent_daily_mean_2_prediction"] = (
                combined["recent_daily_mean_2"] * combined["days_in_month"]
            )
            combined["log_recent_daily_mean_2_prediction"] = _safe_log_external(
                combined["recent_daily_mean_2_prediction"], eps=eps
            )
            _mark("recent_daily_mean_2")
            _mark("recent_daily_mean_2_prediction")
            _mark("log_recent_daily_mean_2_prediction")

        if {daily_1, daily_2, daily_3}.issubset(combined.columns):
            recent_cols = [daily_1, daily_2, daily_3]
            combined["recent_daily_mean_3"] = combined[recent_cols].mean(axis=1)
            combined["recent_daily_median_3"] = combined[recent_cols].median(axis=1)
            combined["recent_daily_weighted_123"] = (
                0.60 * combined[daily_1] + 0.25 * combined[daily_2] + 0.15 * combined[daily_3]
            )

            for col in [
                "recent_daily_mean_3",
                "recent_daily_median_3",
                "recent_daily_weighted_123",
            ]:
                pred_col = f"{col}_prediction"
                combined[pred_col] = combined[col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            combined["daily_growth_1_2"] = _safe_divide_external(
                combined[daily_1], combined[daily_2], fill=np.nan
            )
            combined["daily_growth_1_3"] = _safe_divide_external(
                combined[daily_1], combined[daily_3], fill=np.nan
            )
            combined["daily_growth_1_2_clipped"] = combined["daily_growth_1_2"].clip(0.70, 1.35)
            combined["daily_growth_1_3_clipped"] = combined["daily_growth_1_3"].clip(0.70, 1.35)
            _mark("daily_growth_1_2")
            _mark("daily_growth_1_3")
            _mark("daily_growth_1_2_clipped")
            _mark("daily_growth_1_3_clipped")

            if "day_scaled_lag1_prediction" in combined.columns:
                combined["trend_day_scaled_lag1_prediction"] = (
                    combined["day_scaled_lag1_prediction"] * combined["daily_growth_1_2_clipped"]
                )
                combined["trend_day_scaled_lag1_prediction_smooth"] = combined[
                    "day_scaled_lag1_prediction"
                ] * np.sqrt(combined["daily_growth_1_2_clipped"])
                combined["log_trend_day_scaled_lag1_prediction"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction"], eps=eps
                )
                combined["log_trend_day_scaled_lag1_prediction_smooth"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction_smooth"], eps=eps
                )
                _mark("trend_day_scaled_lag1_prediction")
                _mark("trend_day_scaled_lag1_prediction_smooth")
                _mark("log_trend_day_scaled_lag1_prediction")
                _mark("log_trend_day_scaled_lag1_prediction_smooth")

    if add_transition_anchor_features:
        pto12, pto13 = f"{target_col}_lag_12", f"{target_col}_lag_13"
        daily1, daily12, daily13 = (
            f"daily_{target_col}_lag_1",
            f"daily_{target_col}_lag_12",
            f"daily_{target_col}_lag_13",
        )

        if {pto12, pto13, f"{target_col}_lag_1"}.issubset(combined.columns):
            combined["same_transition_last_year_factor"] = _safe_divide_external(
                combined[pto12], combined[pto13], fill=np.nan
            )
            combined["same_transition_last_year_factor_clipped"] = combined[
                "same_transition_last_year_factor"
            ].clip(0.75, 1.45)
            combined["same_transition_last_year_prediction"] = (
                combined[f"{target_col}_lag_1"]
                * combined["same_transition_last_year_factor_clipped"]
            )
            combined["log_same_transition_last_year_prediction"] = _safe_log_external(
                combined["same_transition_last_year_prediction"], eps=eps
            )
            _mark("same_transition_last_year_factor")
            _mark("same_transition_last_year_factor_clipped")
            _mark("same_transition_last_year_prediction")
            _mark("log_same_transition_last_year_prediction")

        if {daily1, daily12, daily13, "days_in_month"}.issubset(combined.columns):
            combined["daily_same_transition_last_year_factor"] = _safe_divide_external(
                combined[daily12], combined[daily13], fill=np.nan
            )
            combined["daily_same_transition_last_year_factor_clipped"] = combined[
                "daily_same_transition_last_year_factor"
            ].clip(0.75, 1.35)
            combined["daily_same_transition_last_year_prediction"] = (
                combined[daily1]
                * combined["daily_same_transition_last_year_factor_clipped"]
                * combined["days_in_month"]
            )
            combined["log_daily_same_transition_last_year_prediction"] = _safe_log_external(
                combined["daily_same_transition_last_year_prediction"], eps=eps
            )
            _mark("daily_same_transition_last_year_factor")
            _mark("daily_same_transition_last_year_factor_clipped")
            _mark("daily_same_transition_last_year_prediction")
            _mark("log_daily_same_transition_last_year_prediction")

            combined["global_daily_transition_factor"] = combined.groupby(time_col, dropna=False)[
                "daily_same_transition_last_year_factor_clipped"
            ].transform("median")
            combined["global_transition_prediction"] = (
                combined[daily1]
                * combined["global_daily_transition_factor"]
                * combined["days_in_month"]
            )
            combined["log_global_transition_prediction"] = _safe_log_external(
                combined["global_transition_prediction"], eps=eps
            )
            _mark("global_daily_transition_factor")
            _mark("global_transition_prediction")
            _mark("log_global_transition_prediction")

            if group_cols_for_transition is None:
                group_cols_for_transition = []
                for col in [
                    "region",
                    "trading_area_category",
                    "opening_date_category",
                    "settlement",
                ]:
                    if col in combined.columns:
                        group_cols_for_transition.append([col])
                if {"region", "trading_area_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "trading_area_category"])
                if {"region", "opening_date_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "opening_date_category"])

            for group_cols in group_cols_for_transition:
                existing = [c for c in group_cols if c in combined.columns]
                if not existing:
                    continue
                group_name = "__".join(existing)
                factor_col = f"{group_name}_daily_transition_factor"
                pred_col = f"{group_name}_transition_prediction"
                combined[factor_col] = combined.groupby([time_col] + existing, dropna=False)[
                    "daily_same_transition_last_year_factor_clipped"
                ].transform("median")
                combined[pred_col] = (
                    combined[daily1] * combined[factor_col] * combined["days_in_month"]
                )
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(factor_col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

        # Blends
        blend_sources = {
            "blend_day_scaled_and_last_year_prediction": (
                "day_scaled_lag1_prediction",
                "daily_same_transition_last_year_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_global_prediction": (
                "day_scaled_lag1_prediction",
                "global_transition_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_recent_mean_prediction": (
                "day_scaled_lag1_prediction",
                "recent_daily_mean_3_prediction",
                0.60,
                0.40,
            ),
        }
        for out_col, (c1, c2, w1, w2) in blend_sources.items():
            if {c1, c2}.issubset(combined.columns):
                combined[out_col] = w1 * combined[c1] + w2 * combined[c2]
                combined[f"log_{out_col}"] = _safe_log_external(combined[out_col], eps=eps)
                _mark(out_col)
                _mark(f"log_{out_col}")

        ratio_pairs = {
            "day_scaled_to_lag1_ratio": ("day_scaled_lag1_prediction", f"{target_col}_lag_1"),
            "last_year_anchor_to_day_scaled_ratio": (
                "daily_same_transition_last_year_prediction",
                "day_scaled_lag1_prediction",
            ),
            "global_anchor_to_day_scaled_ratio": (
                "global_transition_prediction",
                "day_scaled_lag1_prediction",
            ),
            "recent_mean_anchor_to_day_scaled_ratio": (
                "recent_daily_mean_3_prediction",
                "day_scaled_lag1_prediction",
            ),
        }
        for out_col, (a, b) in ratio_pairs.items():
            if {a, b}.issubset(combined.columns):
                combined[out_col] = _safe_divide_external(combined[a], combined[b], fill=np.nan)
                _mark(out_col)

        log_diff_pairs = [(1, 2), (1, 3), (1, 6), (1, 12), (2, 3), (3, 6), (12, 13)]
        for left, right in log_diff_pairs:
            a, b = f"{target_col}_lag_{left}", f"{target_col}_lag_{right}"
            if {a, b}.issubset(combined.columns):
                out_col = f"log_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[a], eps=eps) - _safe_log_external(
                    combined[b], eps=eps
                )
                _mark(out_col)

            da, db = f"daily_{target_col}_lag_{left}", f"daily_{target_col}_lag_{right}"
            if {da, db}.issubset(combined.columns):
                out_col = f"log_daily_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[da], eps=eps) - _safe_log_external(
                    combined[db], eps=eps
                )
                _mark(out_col)

    # ------------------------------------------------------------
    # 3. March 2024 growth coefficients
    # ------------------------------------------------------------
    if add_march_2024_features:
        march_2024_idx = (2024 - int(min_year)) * 12 + (3 - 1)
        after_march_2024 = combined[time_col].astype(int) > march_2024_idx

        hist = history_raw.copy()
        hist["calendar_month_num"] = _calendar_month_from_time(hist[time_col], min_year)
        hist["calendar_year"] = _year_from_time(hist[time_col], min_year)

        hist_2024 = hist[
            (hist["calendar_year"].astype(int) == 2024)
            & (hist["calendar_month_num"].isin([2, 3]))
            & hist[target_col].notna()
        ].copy()

        if not hist_2024.empty:
            store_pivot = (
                hist_2024.pivot_table(
                    index=id_col,
                    columns="calendar_month_num",
                    values=target_col,
                    aggfunc="first",
                )
                .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                .reset_index()
            )
            if {"pto_feb_2024", "pto_mar_2024"}.issubset(store_pivot.columns):
                store_pivot["store_feb_mar_growth_2024"] = _safe_divide_external(
                    store_pivot["pto_mar_2024"], store_pivot["pto_feb_2024"], fill=np.nan
                ).clip(clip_growth_low, clip_growth_high)
                combined = combined.drop(columns=["store_feb_mar_growth_2024"], errors="ignore")
                combined = combined.merge(
                    store_pivot[[id_col, "store_feb_mar_growth_2024"]],
                    on=id_col,
                    how="left",
                )
                combined.loc[~after_march_2024, "store_feb_mar_growth_2024"] = np.nan
                _mark("store_feb_mar_growth_2024")

            def _add_group_growth(group_col, out_col):
                nonlocal combined
                if group_col not in hist_2024.columns or group_col not in combined.columns:
                    return
                group_agg = (
                    hist_2024.groupby([group_col, "calendar_month_num"], dropna=False)[target_col]
                    .sum()
                    .reset_index()
                )
                pivot = (
                    group_agg.pivot_table(
                        index=group_col,
                        columns="calendar_month_num",
                        values=target_col,
                        aggfunc="first",
                    )
                    .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                    .reset_index()
                )
                if {"pto_feb_2024", "pto_mar_2024"}.issubset(pivot.columns):
                    pivot[out_col] = _safe_divide_external(
                        pivot["pto_mar_2024"], pivot["pto_feb_2024"], fill=np.nan
                    ).clip(clip_growth_low, clip_growth_high)
                    combined = combined.drop(columns=[out_col], errors="ignore")
                    combined = combined.merge(pivot[[group_col, out_col]], on=group_col, how="left")
                    combined.loc[~after_march_2024, out_col] = np.nan
                    _mark(out_col)

            _add_group_growth("region", "region_feb_mar_growth_2024")
            _add_group_growth("settlement", "settlement_feb_mar_growth_2024")
            _add_group_growth("trading_area_category", "trading_area_feb_mar_growth_2024")

            feb_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 2, target_col].sum()
            mar_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 3, target_col].sum()
            global_growth = np.nan
            if pd.notna(feb_sum) and feb_sum > 0:
                global_growth = np.clip(mar_sum / feb_sum, clip_growth_low, clip_growth_high)

            combined["global_feb_mar_growth_2024"] = np.where(
                after_march_2024, global_growth, np.nan
            )
            _mark("global_feb_mar_growth_2024")

            growth_cols = [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_category_feb_mar_growth_2024",  # backward-safe typo guard
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]
            # unify expected column name from the prompt
            if (
                "trading_area_category_feb_mar_growth_2024" in combined.columns
                and "trading_area_feb_mar_growth_2024" not in combined.columns
            ):
                combined["trading_area_feb_mar_growth_2024"] = combined[
                    "trading_area_category_feb_mar_growth_2024"
                ]
                _mark("trading_area_feb_mar_growth_2024")

            for col in [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]:
                if col not in combined.columns or f"{target_col}_lag_1" not in combined.columns:
                    continue
                combined[col] = combined[col].fillna(combined["global_feb_mar_growth_2024"])
                pred_col = f"lag1_x_{col}"
                combined[pred_col] = combined[f"{target_col}_lag_1"] * combined[col]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            if {"day_scaled_lag1_prediction", "lag1_x_store_feb_mar_growth_2024"}.issubset(
                combined.columns
            ):
                combined["blend_day_scaled_and_store_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_store_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_store_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_store_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_store_march_growth")
                _mark("log_blend_day_scaled_and_store_march_growth")

            if {"day_scaled_lag1_prediction", "lag1_x_region_feb_mar_growth_2024"}.issubset(
                combined.columns
            ):
                combined["blend_day_scaled_and_region_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_region_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_region_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_region_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_region_march_growth")
                _mark("log_blend_day_scaled_and_region_march_growth")

    # ------------------------------------------------------------
    # 4. Hierarchical share features
    # ------------------------------------------------------------
    if add_hierarchy_share_features:
        pto_lag_1 = f"{target_col}_lag_1"
        if pto_lag_1 in combined.columns:
            if "region" in combined.columns:
                combined["region_pto_lag1_mean"] = combined.groupby(
                    [time_col, "region"], dropna=False
                )[pto_lag_1].transform("mean")
                combined["store_share_region_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["region_pto_lag1_mean"], fill=np.nan
                )
                _mark("region_pto_lag1_mean")
                _mark("store_share_region_lag1")

            if "settlement" in combined.columns:
                combined["settlement_pto_lag1_mean"] = combined.groupby(
                    [time_col, "settlement"], dropna=False
                )[pto_lag_1].transform("mean")
                combined["store_share_settlement_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["settlement_pto_lag1_mean"], fill=np.nan
                )
                _mark("settlement_pto_lag1_mean")
                _mark("store_share_settlement_lag1")

            if "trading_area_category" in combined.columns:
                combined["area_category_pto_lag1_mean"] = combined.groupby(
                    [time_col, "trading_area_category"], dropna=False
                )[pto_lag_1].transform("mean")
                combined["store_share_area_category_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["area_category_pto_lag1_mean"], fill=np.nan
                )
                _mark("area_category_pto_lag1_mean")
                _mark("store_share_area_category_lag1")

            if "region" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"region_pto_lag{lag}_mean"
                    share_col = f"store_share_region_lag{lag}"
                    combined[mean_col] = combined.groupby([time_col, "region"], dropna=False)[
                        lag_col
                    ].transform("mean")
                    combined[share_col] = _safe_divide_external(
                        combined[lag_col], combined[mean_col], fill=np.nan
                    )
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [
                    c
                    for c in [
                        "store_share_region_lag1",
                        "store_share_region_lag2",
                        "store_share_region_lag3",
                    ]
                    if c in combined.columns
                ]
                if share_cols:
                    combined["store_share_region_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_region_mean_3")
                if {"store_share_region_lag1", "store_share_region_lag2"}.issubset(
                    combined.columns
                ):
                    combined["store_share_region_change_1_2"] = _safe_divide_external(
                        combined["store_share_region_lag1"],
                        combined["store_share_region_lag2"],
                        fill=np.nan,
                    )
                    combined["store_share_region_change_1_2_clipped"] = combined[
                        "store_share_region_change_1_2"
                    ].clip(0.70, 1.35)
                    _mark("store_share_region_change_1_2")
                    _mark("store_share_region_change_1_2_clipped")

            if "settlement" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"settlement_pto_lag{lag}_mean"
                    share_col = f"store_share_settlement_lag{lag}"
                    combined[mean_col] = combined.groupby([time_col, "settlement"], dropna=False)[
                        lag_col
                    ].transform("mean")
                    combined[share_col] = _safe_divide_external(
                        combined[lag_col], combined[mean_col], fill=np.nan
                    )
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [
                    c
                    for c in [
                        "store_share_settlement_lag1",
                        "store_share_settlement_lag2",
                        "store_share_settlement_lag3",
                    ]
                    if c in combined.columns
                ]
                if share_cols:
                    combined["store_share_settlement_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_settlement_mean_3")

            if {"store_share_region_mean_3", "lag1_x_region_feb_mar_growth_2024"}.issubset(
                combined.columns
            ):
                combined["region_anchor_x_store_share_mean_3"] = combined[
                    "lag1_x_region_feb_mar_growth_2024"
                ] * combined["store_share_region_mean_3"].clip(0.50, 1.80)
                combined["log_region_anchor_x_store_share_mean_3"] = _safe_log_external(
                    combined["region_anchor_x_store_share_mean_3"], eps=eps
                )
                _mark("region_anchor_x_store_share_mean_3")
                _mark("log_region_anchor_x_store_share_mean_3")

            if {"store_share_region_change_1_2_clipped", "day_scaled_lag1_prediction"}.issubset(
                combined.columns
            ):
                combined["day_scaled_x_region_share_change"] = (
                    combined["day_scaled_lag1_prediction"]
                    * combined["store_share_region_change_1_2_clipped"]
                )
                combined["log_day_scaled_x_region_share_change"] = _safe_log_external(
                    combined["day_scaled_x_region_share_change"], eps=eps
                )
                _mark("day_scaled_x_region_share_change")
                _mark("log_day_scaled_x_region_share_change")

    # ------------------------------------------------------------
    # 5. Operational lag features
    # ------------------------------------------------------------
    if add_operational_lag_features:
        # lag_1, lag_3, lag_12 из исходных operational cols
        for col in lagged_monthly_cols:
            if col not in history_raw.columns:
                continue
            for lag in [1, 3, 12]:
                _merge_history_lag(history_raw, col, lag, f"{col}_lag_{lag}", [id_col])

            # rolling mean/median по прошлым 3 наблюдениям.
            raw_combined = pd.concat(
                [
                    history_raw.assign(__part_roll__="history"),
                    future_raw.assign(__part_roll__="future"),
                ],
                axis=0,
                ignore_index=True,
                sort=False,
            )
            raw_combined = _ensure_time_year_month_external(
                raw_combined,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                min_year=min_year,
                name=f"raw_combined_for_{col}",
            )
            raw_combined = raw_combined.sort_values([id_col, time_col], kind="mergesort")
            raw_combined["__x__"] = np.where(
                raw_combined["__part_roll__"].eq("history"), raw_combined[col], np.nan
            )
            shifted = raw_combined.groupby(id_col, sort=False)["__x__"].shift(1)
            raw_combined[f"{col}_roll_mean_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            raw_combined[f"{col}_roll_median_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .median()
                .reset_index(level=0, drop=True)
            )
            roll_map = raw_combined[
                [id_col, time_col, f"{col}_roll_mean_3", f"{col}_roll_median_3"]
            ].drop_duplicates(
                [id_col, time_col],
                keep="last",
            )
            combined = combined.drop(
                columns=[f"{col}_roll_mean_3", f"{col}_roll_median_3"], errors="ignore"
            )
            combined = combined.merge(roll_map, on=[id_col, time_col], how="left")
            _mark(f"{col}_roll_mean_3")
            _mark(f"{col}_roll_median_3")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1"}.issubset(combined.columns):
            combined["pto_per_working_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"],
                combined["working_hours_per_day_lag_1"],
                fill=np.nan,
            )
            combined["log_pto_per_working_hour_lag1"] = _safe_log_external(
                combined["pto_per_working_hour_lag1"], eps=eps
            )
            _mark("pto_per_working_hour_lag1")
            _mark("log_pto_per_working_hour_lag1")

        if {f"{target_col}_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            combined["pto_per_cash_register_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"],
                combined["number_of_cash_registers"].astype(float),
                fill=np.nan,
            )
            combined["log_pto_per_cash_register_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_lag1")
            _mark("log_pto_per_cash_register_lag1")

        if {
            f"{target_col}_lag_1",
            "working_hours_per_day_lag_1",
            "number_of_cash_registers",
        }.issubset(combined.columns):
            denom = combined["working_hours_per_day_lag_1"] * combined[
                "number_of_cash_registers"
            ].astype(float)
            combined["pto_per_cash_register_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], denom, fill=np.nan
            )
            combined["log_pto_per_cash_register_hour_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_hour_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_hour_lag1")
            _mark("log_pto_per_cash_register_hour_lag1")

    # ------------------------------------------------------------
    # Final cleanup and restore parts
    # ------------------------------------------------------------
    created = sorted(set(c for c in created if c in combined.columns))
    combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)

    if fill_history_na and created:
        combined[created] = combined[created].fillna(fill_value)

    combined = combined.drop(columns=["calendar_year"], errors="ignore")

    history_out = (
        combined[combined["__part__"] == "history"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )
    future_out = (
        combined[combined["__part__"] == "future"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )

    # Align columns
    ordered_cols = list(history_out.columns)
    for c in future_out.columns:
        if c not in ordered_cols:
            ordered_cols.append(c)
    for c in ordered_cols:
        if c not in history_out.columns:
            history_out[c] = np.nan
        if c not in future_out.columns:
            future_out[c] = np.nan
    history_out = history_out[ordered_cols]
    future_out = future_out[ordered_cols]

    if downcast_float_features:
        for df_ in (history_out, future_out):
            float_cols = [
                c for c in df_.select_dtypes(include=["float64"]).columns if c != target_col
            ]
            if float_cols:
                df_[float_cols] = df_[float_cols].astype("float32")

    return history_out, future_out, created


def build_retail_features_with_split(
    df: pd.DataFrame,
    *,
    # split config
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    create_test_if_missing: bool = True,
    expected_test_size: Optional[int] = 18657,
    future_unknown_cols: Optional[Sequence[str]] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    # feature config
    use_val_as_test_history: bool = True,
    add_integrated_anchor_features: bool = True,
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    # pass-through to base generator
    base_feature_kwargs: dict | None = None,
    verbose: bool = True,
) -> dict:
    """
    Единая функция:
    1) делает leakage-safe split;
    2) создаёт artificial test month, если нужно;
    3) запускает базовый generate_retail_forecast_features_leakfree;
    4) добавляет все новые идеи:
       - month_day_ratio_to_previous_month;
       - day_scaled_lag1_prediction;
       - expanding calendar heuristic factors/predictions;
       - daily_pto_lag_*;
       - recent daily baselines;
       - trend-corrected anchors;
       - same-transition-last-year anchors;
       - global/group transition anchors;
       - March 2024 growth coefficients;
       - hierarchical share features;
       - lagged operational features.
    """
    base_feature_kwargs = {} if base_feature_kwargs is None else dict(base_feature_kwargs)
    # Defaults chosen to cover the full feature-idea corpus. User can override them in base_feature_kwargs.
    base_feature_kwargs.setdefault("target_rolling_windows", (3, 6, 12))
    base_feature_kwargs.setdefault("monthly_cross_lags", (1, 3, 12))
    base_feature_kwargs.setdefault(
        "monthly_cross_feature_types",
        (
            "promo_share",
            "promo_x_items",
            "items_x_hours",
            "cancellations_per_item",
            "cancellations_per_hour",
        ),
    )

    split_result = make_panel_time_split(
        df,
        group_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        split_mode=split_mode,
        val_time_idx=val_time_idx,
        val_year=val_year,
        val_month=val_month,
        test_time_idx=test_time_idx,
        test_year=test_year,
        test_month=test_month,
        min_year=min_year,
        create_test_if_missing=create_test_if_missing,
        future_unknown_cols=future_unknown_cols,
        expected_test_size=expected_test_size
        if split_mode in {"train_val_test", "train_test"}
        else None,
        verbose=verbose,
    )

    if split_mode == "train_val":
        train_df, val_df = split_result
        test_df = None
    elif split_mode == "train_val_test":
        train_df, val_df, test_df = split_result
    else:
        train_df, test_df = split_result
        val_df = None

    output = generate_retail_forecast_features_leakfree(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        use_val_as_test_history=use_val_as_test_history,
        verbose=verbose,
        **base_feature_kwargs,
    )

    output["raw_splits"] = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    if add_integrated_anchor_features:
        # validation pair: history = train, future = val
        if output.get("train_features") is not None and output.get("val_features") is not None:
            train_feat, val_feat, created = _add_integrated_anchor_features_pair(
                output["train_features"],
                output["val_features"],
                train_df,
                val_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                train_feat = train_feat.drop(columns=[year_col], errors="ignore")
                val_feat = val_feat.drop(columns=[year_col], errors="ignore")
            output["train_features"] = train_feat
            output["val_features"] = val_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})[
                "val_created"
            ] = created
            if verbose:
                print("Added integrated anchor features to train/val:", len(created))

        # test pair: history = train + val if requested and val exists, otherwise train
        if (
            output.get("test_train_features") is not None
            and output.get("test_features") is not None
        ):
            if val_df is not None and use_val_as_test_history:
                test_history_raw = pd.concat(
                    [train_df, val_df], axis=0, ignore_index=True, sort=False
                )
            else:
                test_history_raw = train_df.copy()

            test_train_feat, test_feat, created = _add_integrated_anchor_features_pair(
                output["test_train_features"],
                output["test_features"],
                test_history_raw,
                test_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                test_train_feat = test_train_feat.drop(columns=[year_col], errors="ignore")
                test_feat = test_feat.drop(columns=[year_col], errors="ignore")
            output["test_train_features"] = test_train_feat
            output["test_features"] = test_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})[
                "test_created"
            ] = created
            if verbose:
                print("Added integrated anchor features to train/test:", len(created))

    # Recompute model columns after enhancement.
    ref = (
        output["test_train_features"]
        if output.get("test_train_features") is not None
        else output.get("train_features")
    )
    if ref is not None:
        exclude = {target_col}
        output["feature_cols"] = [c for c in ref.columns if c not in exclude]
        output["categorical_cols"] = [
            c
            for c in output["feature_cols"]
            if c in ref.columns
            and (pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c]))
        ]

    gc.collect()
    return output


import numpy as np
import pandas as pd
from typing import Optional, Sequence, Literal


def prepare_lgbm_matrices_with_time_aware_catboost_encoding(
    feature_output: dict,
    *,
    features_to_use: Optional[Sequence[str]] = None,
    target_mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",
    target_col: str = "pto",
    id_col: str = "new_id",
    time_col: str = "year_month_index",
    cat_features: Optional[Sequence[str]] = None,
    add_default_cat_features: bool = True,
    add_numeric_cat_copies: bool = True,
    numeric_cat_source_cols: Sequence[str] = (
        "new_id",
        "alcohol_license_flag",
        "year",
        "month",
        "prev_month",
        "calendar_month_num",
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
        "working_hours_per_day_lag_1",
        "working_hours_per_day_lag_3",
        "working_hours_per_day_lag_12",
    ),
    cb_suffix: str = "_cb",
    smoothing: float = 20.0,
    min_count: int = 1,
    drop_original_cat_features: bool = True,
    drop_numeric_cat_source_cols: bool = False,
    include_all_detected_cat_features: bool = True,
    keep_time_col: bool = True,
    keep_id_col: bool = True,
    strict_train_time_order: bool = True,
    first_block_value: Literal["nan", "global"] = "nan",
    unknown_value: Literal["global", "nan"] = "global",
    verbose: bool = True,
) -> dict:
    """
    Готовит матрицы для LightGBM/числовой модели с time-aware ordered target encoding
    для всех категориальных признаков.

    Основной контракт против leakage:
    - validation: encoder fit только на train_features; val только transform;
    - test: encoder fit только на test_train_features; test только transform;
    - train-строка месяца t кодируется только статистиками месяцев < t;
    - same-month target никогда не используется для кодировки train-строк этого же месяца;
    - fallback prior внутри train также time-aware: для месяца t берётся среднее только по месяцам < t;
    - для первого train-блока fallback по умолчанию NaN, а не mean по всему train.

    Почему это не нативный CatBoost:
    - это ручной CatBoost-like / ordered target encoding;
    - на выходе не остаётся object/category/string колонок;
    - исходные категориальные признаки по умолчанию удаляются, остаются *_cb.

    Важные параметры:
    - include_all_detected_cat_features=True добавляет в признаки все найденные
      категориальные колонки, даже если features_to_use их не содержал;
    - first_block_value="nan" максимально строго исключает temporal leakage;
      если нужна менее строгая, но более плотная матрица, можно поставить "global";
    - unknown_value="global" для val/test безопасен, потому что global считается
      только по fit-history.
    """

    # ------------------------------------------------------------
    # Basic checks
    # ------------------------------------------------------------
    if smoothing < 0:
        raise ValueError("smoothing должен быть >= 0.")
    if min_count < 1:
        raise ValueError("min_count должен быть >= 1.")
    if first_block_value not in {"nan", "global"}:
        raise ValueError("first_block_value должен быть 'nan' или 'global'.")
    if unknown_value not in {"global", "nan"}:
        raise ValueError("unknown_value должен быть 'global' или 'nan'.")
    if not strict_train_time_order:
        raise ValueError(
            "strict_train_time_order=False отключал бы защиту от temporal leakage. "
            "Для этой задачи оставляем только strict_train_time_order=True."
        )

    def _check_required_functions():
        if "make_forecast_target" not in globals():
            raise NameError(
                "Не найдена функция make_forecast_target. "
                "Она должна быть определена до prepare_lgbm_matrices_with_time_aware_catboost_encoding."
            )

    def _copy_df(df):
        return df.copy() if df is not None else None

    def _is_non_numeric_dtype(s: pd.Series) -> bool:
        return (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_categorical_dtype(s)
            or pd.api.types.is_string_dtype(s)
        )

    def _as_cat_string(s: pd.Series) -> pd.Series:
        # Единая нормализация категорий для train/future.
        # Все пропуски и строковые варианты NA схлопываются в один токен.
        out = s.astype("string")
        out = out.fillna("__missing__")
        out = out.replace(
            {
                "<NA>": "__missing__",
                "nan": "__missing__",
                "NaN": "__missing__",
                "None": "__missing__",
                "none": "__missing__",
                "NULL": "__missing__",
                "null": "__missing__",
            }
        )
        return out

    def _safe_numeric_frame(df: pd.DataFrame, *, name: str) -> pd.DataFrame:
        bad_cols = [c for c in df.columns if _is_non_numeric_dtype(df[c])]
        if bad_cols:
            raise ValueError(
                f"[{name}] После ordered target encoding остались нечисловые колонки: "
                f"{bad_cols[:50]}. Обычно это значит, что колонка не попала в cat_features "
                "или drop_original_cat_features=False."
            )
        return df

    def _ordered_unique(items) -> list:
        seen = set()
        result = []
        for x in items:
            if x is None:
                continue
            if x not in seen:
                seen.add(x)
                result.append(x)
        return result

    # ------------------------------------------------------------
    # Categorical feature construction / discovery
    # ------------------------------------------------------------
    def _add_numeric_copies_to_parts(parts: dict[str, Optional[pd.DataFrame]]):
        """
        Создаёт *_cat копии для числовых/дискретных признаков.
        Если source есть в train, но отсутствует в future, future получает __missing__.
        Это важно, чтобы transform не падал на разных наборах колонок.
        """
        created = []
        if not add_numeric_cat_copies:
            return {k: _copy_df(v) for k, v in parts.items()}, created

        out_parts = {k: _copy_df(v) for k, v in parts.items()}

        for src in numeric_cat_source_cols:
            dst = f"{src}_cat"
            exists_somewhere = any(
                df_part is not None and (src in df_part.columns or dst in df_part.columns)
                for df_part in out_parts.values()
            )
            if not exists_somewhere:
                continue

            for _, df_part in out_parts.items():
                if df_part is None:
                    continue
                if dst in df_part.columns:
                    df_part[dst] = _as_cat_string(df_part[dst])
                elif src in df_part.columns:
                    df_part[dst] = _as_cat_string(df_part[src])
                else:
                    df_part[dst] = "__missing__"

            created.append(dst)

        return out_parts, sorted(set(created))

    def _default_cat_candidates() -> list[str]:
        return [
            "opening_date_category",
            "trading_area_category",
            "settlement",
            "region",
            "month_name",
            "holiday_month_type",
            "population_city_type",
            "season",
            "lag1_rto_size_bin",
            "new_id_cat",
            "alcohol_license_flag_cat",
            "year_cat",
            "month_cat",
            "prev_month_cat",
            "calendar_month_num_cat",
            "marketplaces_deliveries_parcel_lockers_100m_cat",
            "medical_institutions_and_pharmacies_300m_cat",
            "schools_300m_cat",
            "public_transport_stops_300m_cat",
            "grocery_stores_500m_cat",
            "pyaterochka_stores_500m_cat",
            "number_of_cash_registers_cat",
            "working_hours_per_day_lag_1_cat",
            "working_hours_per_day_lag_3_cat",
            "working_hours_per_day_lag_12_cat",
        ]

    def _collect_all_cat_candidates(
        ref_df: pd.DataFrame, numeric_cat_copies: list[str]
    ) -> list[str]:
        candidates = []
        if cat_features is not None:
            candidates.extend(list(cat_features))
        candidates.extend(feature_output.get("categorical_cols") or [])
        candidates.extend([c for c in ref_df.columns if _is_non_numeric_dtype(ref_df[c])])
        candidates.extend(numeric_cat_copies)
        if add_default_cat_features:
            candidates.extend(_default_cat_candidates())
        return [c for c in _ordered_unique(candidates) if c in ref_df.columns and c != target_col]

    def _build_base_feature_list(
        train_df: pd.DataFrame, numeric_cat_copies: list[str]
    ) -> list[str]:
        if features_to_use is None:
            base_features = list(feature_output.get("feature_cols") or [])
            if not base_features:
                base_features = [c for c in train_df.columns if c != target_col]
        else:
            base_features = list(features_to_use)

        base_features = [c for c in base_features if c != target_col]
        base_features = [c for c in base_features if c in train_df.columns]

        # Newly created *_cat должны попасть в список признаков, иначе encoding не будет создан.
        for c in numeric_cat_copies:
            if c in train_df.columns and c not in base_features:
                base_features.append(c)

        # По требованию: все обнаруженные категориальные признаки пропускаем через ordered TE.
        # Если features_to_use был строгим selected-list, это поведение можно выключить
        # include_all_detected_cat_features=False.
        if include_all_detected_cat_features:
            for c in _collect_all_cat_candidates(train_df, numeric_cat_copies):
                if c in train_df.columns and c not in base_features:
                    base_features.append(c)

        return _ordered_unique(base_features)

    def _collect_cat_features(
        ref_df: pd.DataFrame, feature_cols: list[str], numeric_cat_copies: list[str]
    ) -> list[str]:
        feature_set = set(feature_cols)
        candidates = _collect_all_cat_candidates(ref_df, numeric_cat_copies)
        result = []
        for c in candidates:
            if c in ref_df.columns and c in feature_set and c not in result:
                result.append(c)
        return result

    # ------------------------------------------------------------
    # Ordered target encoding core
    # ------------------------------------------------------------
    def _smoothed_value(sum_y: float, cnt: int, prior: float) -> float:
        if cnt < min_count:
            return prior
        return (float(sum_y) + float(smoothing) * float(prior)) / (int(cnt) + float(smoothing))

    def _make_future_encoding(
        future_col: pd.Series,
        stats: pd.DataFrame,
        *,
        global_prior: float,
    ) -> pd.Series:
        cats = _as_cat_string(future_col)

        if stats.empty:
            fill = global_prior if unknown_value == "global" else np.nan
            return pd.Series(fill, index=future_col.index, dtype="float32")

        sums = cats.map(stats["sum"]).astype(float)
        cnts = cats.map(stats["count"]).astype(float)

        if unknown_value == "global":
            prior = global_prior
        else:
            prior = np.nan

        enc = (sums + smoothing * global_prior) / (cnts + smoothing)
        enc = enc.astype(float)
        enc = enc.where(cnts.fillna(0) >= min_count, prior)
        enc = enc.fillna(prior)
        return enc.astype("float32")

    def _time_aware_cb_encode_pair(
        train_df: pd.DataFrame,
        future_df: Optional[pd.DataFrame],
        y_train: pd.Series,
        *,
        feature_cols: list[str],
        cat_cols: list[str],
        pair_name: str,
    ):
        """
        Block-wise ordered target encoding:
        - train month t uses only rows with time < t;
        - fallback prior for train month t is also computed only from time < t;
        - future uses all train rows, which are historical relative to that future block.
        """
        train_df = train_df.copy()
        future_df = future_df.copy() if future_df is not None else None

        if time_col not in train_df.columns:
            raise ValueError(f"[{pair_name}] В train_df нет time_col='{time_col}'.")
        if train_df[time_col].isna().any():
            raise ValueError(f"[{pair_name}] В train_df['{time_col}'] есть NaN.")

        y_train = pd.Series(y_train, index=train_df.index)
        y_train = pd.to_numeric(y_train, errors="coerce").astype(float)
        valid_y_mask = y_train.notna() & np.isfinite(y_train)
        if valid_y_mask.sum() == 0:
            raise ValueError(f"[{pair_name}] y_train полностью NaN/inf после make_forecast_target.")

        # Full-history mean безопасен только для future transform.
        full_history_global_mean = float(y_train.loc[valid_y_mask].mean())

        # Гарантируем наличие cat_cols в train/future. Если в future нет колонки – это unseen/missing категория.
        for col in cat_cols:
            if col not in train_df.columns:
                train_df[col] = "__missing__"
            train_df[col] = _as_cat_string(train_df[col])
            if future_df is not None:
                if col not in future_df.columns:
                    future_df[col] = "__missing__"
                future_df[col] = _as_cat_string(future_df[col])

        encoded_cols = []
        ordered_times = sorted(
            pd.to_numeric(train_df[time_col], errors="raise").astype(int).unique()
        )

        # Предрасчёт индексов по времени. Так меньше риска случайно использовать same-month target.
        time_values = pd.to_numeric(train_df[time_col], errors="raise").astype(int)
        indices_by_time = {t: train_df.index[time_values.eq(t)] for t in ordered_times}

        for col in cat_cols:
            enc_col = f"{col}{cb_suffix}"
            encoded_cols.append(enc_col)
            train_df[enc_col] = np.nan

            sum_by_cat: dict[str, float] = {}
            cnt_by_cat: dict[str, int] = {}
            global_sum = 0.0
            global_cnt = 0

            for t in ordered_times:
                idx_t = indices_by_time[t]
                cats_t = train_df.loc[idx_t, col]

                if global_cnt > 0:
                    time_prior = global_sum / global_cnt
                elif first_block_value == "global":
                    # Не идеально строго, но оставлено как управляемый режим совместимости.
                    time_prior = full_history_global_mean
                else:
                    # Максимально строгий режим: нет прошлой истории -> NaN.
                    time_prior = np.nan

                vals = []
                for cat in cats_t:
                    n = cnt_by_cat.get(cat, 0)
                    if n < min_count or pd.isna(time_prior):
                        vals.append(time_prior)
                    else:
                        vals.append(_smoothed_value(sum_by_cat.get(cat, 0.0), n, time_prior))

                train_df.loc[idx_t, enc_col] = vals

                # Обновляем статистики только после кодирования всего time-блока.
                y_t = y_train.loc[idx_t]
                valid_t = y_t.notna() & np.isfinite(y_t)
                if valid_t.any():
                    tmp = pd.DataFrame(
                        {
                            "__cat__": cats_t.loc[valid_t].values,
                            "__y__": y_t.loc[valid_t].values,
                        }
                    )
                    agg = tmp.groupby("__cat__", dropna=False)["__y__"].agg(["sum", "count"])

                    for cat, row in agg.iterrows():
                        sum_by_cat[cat] = sum_by_cat.get(cat, 0.0) + float(row["sum"])
                        cnt_by_cat[cat] = cnt_by_cat.get(cat, 0) + int(row["count"])

                    global_sum += float(y_t.loc[valid_t].sum())
                    global_cnt += int(valid_t.sum())

            train_df[enc_col] = pd.to_numeric(train_df[enc_col], errors="coerce").astype("float32")

        # Full train history stats for future transform. This is safe because future is later than fit history.
        full_stats = {}
        for col in cat_cols:
            tmp = pd.DataFrame(
                {
                    "__cat__": train_df.loc[valid_y_mask, col].values,
                    "__y__": y_train.loc[valid_y_mask].values,
                }
            )
            full_stats[col] = tmp.groupby("__cat__", dropna=False)["__y__"].agg(["sum", "count"])

        if future_df is not None:
            for col in cat_cols:
                enc_col = f"{col}{cb_suffix}"
                future_df[enc_col] = _make_future_encoding(
                    future_df[col],
                    full_stats[col],
                    global_prior=full_history_global_mean,
                )

        final_features = list(feature_cols)

        # Добавляем созданные encoded-признаки.
        for enc_col in encoded_cols:
            if enc_col not in final_features:
                final_features.append(enc_col)

        if drop_original_cat_features:
            cat_set = set(cat_cols)
            final_features = [c for c in final_features if c not in cat_set]

        if drop_numeric_cat_source_cols and add_numeric_cat_copies:
            source_cols_to_drop = set()
            for src in numeric_cat_source_cols:
                if f"{src}_cat" in cat_cols:
                    source_cols_to_drop.add(src)
            final_features = [c for c in final_features if c not in source_cols_to_drop]

        if not keep_time_col:
            final_features = [c for c in final_features if c != time_col]
        if not keep_id_col:
            final_features = [c for c in final_features if c != id_col]

        final_features = _ordered_unique([c for c in final_features if c in train_df.columns])

        X_train = train_df[final_features].copy()
        X_future = None
        if future_df is not None:
            for c in final_features:
                if c not in future_df.columns:
                    future_df[c] = np.nan
            X_future = future_df[final_features].copy()

        X_train = _safe_numeric_frame(X_train, name=f"{pair_name}:X_train")
        if X_future is not None:
            X_future = _safe_numeric_frame(X_future, name=f"{pair_name}:X_future")

        num_cols = X_train.select_dtypes(include=[np.number]).columns
        X_train[num_cols] = X_train[num_cols].replace([np.inf, -np.inf], np.nan)
        if X_future is not None:
            num_cols_f = X_future.select_dtypes(include=[np.number]).columns
            X_future[num_cols_f] = X_future[num_cols_f].replace([np.inf, -np.inf], np.nan)

        # Служебная диагностика: доля NaN в train encoding показывает первый блок и новые категории.
        encoded_nan_share = {}
        for c in encoded_cols:
            if c in X_train.columns:
                encoded_nan_share[c] = float(X_train[c].isna().mean())

        info = {
            "pair_name": pair_name,
            "cat_features_used": cat_cols,
            "encoded_features_created": encoded_cols,
            "final_features": final_features,
            "global_mean_for_future_transform": full_history_global_mean,
            "smoothing": smoothing,
            "min_count": min_count,
            "first_block_value": first_block_value,
            "unknown_value": unknown_value,
            "strict_train_time_order": strict_train_time_order,
            "encoded_train_nan_share": encoded_nan_share,
            "n_train_rows": int(len(train_df)),
            "n_future_rows": int(0 if future_df is None else len(future_df)),
            "n_time_blocks_train": int(len(ordered_times)),
            "first_time_block": int(ordered_times[0]) if ordered_times else None,
            "last_time_block": int(ordered_times[-1]) if ordered_times else None,
        }

        return X_train, X_future, info

    # ------------------------------------------------------------
    # Pair preparation
    # ------------------------------------------------------------
    def _prepare_pair(train_key: str, future_key: Optional[str], pair_name: str):
        train_df = feature_output.get(train_key)
        future_df = feature_output.get(future_key) if future_key is not None else None

        if train_df is None:
            return None

        parts = {
            "train": train_df,
            "future": future_df,
        }
        parts, numeric_cat_copies = _add_numeric_copies_to_parts(parts)
        train_df = parts["train"]
        future_df = parts["future"]

        base_features = _build_base_feature_list(train_df, numeric_cat_copies)
        cat_cols = _collect_cat_features(train_df, base_features, numeric_cat_copies)

        # Если cat_features явно передали, но они не нашлись после feature construction – лучше явно сообщить.
        if cat_features is not None:
            missing_explicit = [c for c in cat_features if c not in train_df.columns]
            if missing_explicit and verbose:
                print(
                    f"[{pair_name}] explicit cat_features отсутствуют в train_df и будут пропущены:",
                    missing_explicit[:30],
                )

        y_train = make_forecast_target(
            train_df,
            mode=target_mode,
            target_col=target_col,
        )

        y_future = None
        if (
            future_df is not None
            and target_col in future_df.columns
            and future_df[target_col].notna().any()
        ):
            y_future = make_forecast_target(
                future_df,
                mode=target_mode,
                target_col=target_col,
            )

        X_train, X_future, info = _time_aware_cb_encode_pair(
            train_df=train_df,
            future_df=future_df,
            y_train=y_train,
            feature_cols=base_features,
            cat_cols=cat_cols,
            pair_name=pair_name,
        )

        return {
            "X_train": X_train,
            "y_train": y_train,
            "X_future": X_future,
            "y_future": y_future,
            "info": info,
        }

    # ------------------------------------------------------------
    # Main
    # ------------------------------------------------------------
    _check_required_functions()

    result = {
        "validation": None,
        "test": None,
        "catboost_encoding_info": {},
    }

    # validation: train -> val
    if (
        feature_output.get("train_features") is not None
        and feature_output.get("val_features") is not None
    ):
        val_pack = _prepare_pair(
            train_key="train_features",
            future_key="val_features",
            pair_name="validation_train_to_val",
        )
        result["validation"] = {
            "X_train": val_pack["X_train"],
            "y_train": val_pack["y_train"],
            "X_val": val_pack["X_future"],
            "y_val": val_pack["y_future"],
        }
        result["catboost_encoding_info"]["validation"] = val_pack["info"]

    # test/final: test_train -> test
    if (
        feature_output.get("test_train_features") is not None
        and feature_output.get("test_features") is not None
    ):
        test_pack = _prepare_pair(
            train_key="test_train_features",
            future_key="test_features",
            pair_name="final_train_to_test",
        )
        result["test"] = {
            "X_train": test_pack["X_train"],
            "y_train": test_pack["y_train"],
            "X_test": test_pack["X_future"],
        }
        result["catboost_encoding_info"]["test"] = test_pack["info"]

    if verbose:
        for block_name, info in result["catboost_encoding_info"].items():
            print(f"\n[catboost-encoding:{block_name}]")
            print("cat features used:", len(info["cat_features_used"]))
            print("first cat features:", info["cat_features_used"][:30])
            print("encoded features created:", len(info["encoded_features_created"]))
            print("first encoded features:", info["encoded_features_created"][:30])
            print("final features:", len(info["final_features"]))
            print("strict train time order:", info["strict_train_time_order"])
            print("first_block_value:", info["first_block_value"])
            print("unknown_value:", info["unknown_value"])

    return result


# ============================================================
# Target engineering utilities
# ============================================================


def make_forecast_target(
    df: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    eps: float = 1e-9,
) -> pd.Series:
    """
    Builds model target for the main target formulations used in the experiments.

    Required feature columns by mode:
    - growth_log: pto_lag_1;
    - day_scaled_residual_log: day_scaled_lag1_prediction;
    - calendar_residual_log: calendar_heuristic_prediction;
    - daily_rate_log: n_days_in_month or days_in_month;
    - direct_log1p: only target_col.
    """
    if target_col not in df.columns:
        raise ValueError(f"В df нет target_col='{target_col}'.")

    y = pd.to_numeric(df[target_col], errors="coerce").astype(float)

    if mode == "original":
        return y

    if mode == "direct_log1p":
        return np.log1p(y.clip(lower=0))

    if mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in df.columns else "days_in_month"
        if day_col not in df.columns:
            raise ValueError(
                "Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'."
            )
        days = pd.to_numeric(df[day_col], errors="coerce").astype(float)
        daily_rate = y / days.replace(0, np.nan)
        return np.log(np.clip(daily_rate, eps, None))

    raise ValueError(f"Неизвестный mode='{mode}'.")


def inverse_forecast_target(
    raw_prediction,
    features: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    residual_shrinkage: float = 1.0,
    clip_min: float = 0.0,
) -> np.ndarray:
    """
    Converts model output back to RTO forecast.

    residual_shrinkage is useful for residual modes, e.g. 0.35 or 0.50.
    """
    pred = np.asarray(raw_prediction, dtype=float)

    if mode == "original":
        out = pred
    elif mode == "direct_log1p":
        out = np.expm1(pred)
    elif mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(pred)
    elif mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in features.columns else "days_in_month"
        if day_col not in features.columns:
            raise ValueError(
                "Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'."
            )
        days = pd.to_numeric(features[day_col], errors="coerce").astype(float).to_numpy()
        out = days * np.exp(pred)
    else:
        raise ValueError(f"Неизвестный mode='{mode}'.")

    out = np.asarray(out, dtype=float)
    out = np.where(np.isfinite(out), out, np.nan)
    if clip_min is not None:
        out = np.maximum(out, clip_min)
    return out

### Признаки четырех моделей

In [ ]:
# ============================================================
# XGBOOST LOG-DIFF FEATURES
# ============================================================

xgboost_log_diff_features = [
    "days_lag_1",
    # near-miss: формально SKIPPED из-за min_improved_folds,
    # но weighted_mape улучшался 5.909577 -> 5.221480;
    # стоит проверить вручную на мартовском holdout
    # "is_halfyear_start_month",
    "is_quarter_end_month",
    "store_share_region_change_1_2",
    "is_year_start_month",
    "pto_lag_1_to_region__opening_date_category_pto_mean_lag_1",
    "pto_per_cash_register_hour_lag1",
    "pto_per_cash_register_lag1",
    "number_of_households",
    "pto_lag_1_to_roll_mean_6",
    "recent_mean_anchor_to_day_scaled_ratio",
    "log_daily_pto_diff_lag_1_2",
    "daily_growth_1_2_clipped",
    "pto_roll_cv_3",
    "pto_lag_1_to_roll_mean_3",
    "is_autumn",
    "holiday_month_type_cb",
    "season_cb",
    "population_size",
    "working_hours_per_day_lag_12_cat_cb",
    "avg_items_per_receipt_diff_lag_1_12",
    "log_pto_diff_lag_6_12",
]


# ============================================================
# LIGHTGBM LOG-DIFF FEATURES
# ============================================================


lightgbm_log_diff_features = [
    "days_lag_1",
    "is_halfyear_start_month",
    "number_of_households",
    "is_year_start_month",
    "days_lag_6",
    "n_working_days",
    "daily_growth_1_2",
    "log_daily_pto_diff_lag_1_2",
    "region_grocery_stores_500m_mean",
    "pto_roll_cv_12",
    "settlement_pto_lag2_mean",
    "pto_lag_1_to_roll_mean_12",
    "pto_lag_1_to_roll_mean_3",
    "log_population_size",
    "n_single_public_holiday_days",
    "region_store_count",
    "households_per_cash_register",
    "region_number_of_cash_registers_mean",
    "pedestrian_traffic_per_1000_people",
    "pto_roll_std_3",
    "is_pre_may_holidays_month",
    "population_per_cash_register",
    "avg_promo_items_per_receipt_roll_cv_3",
    "settlement_pto_lag1_mean",
    "traffic_per_1000_households",
    "log_pto_lag_1",
]


# ============================================================
# LIGHTGBM LOG-LEVEL FEATURES
# ============================================================

lightgbm_log_level_features = [
    "daily_pto_lag_1",
    "log_day_scaled_lag1_prediction",
    "day_scaled_lag1_prediction",
    "is_year_start_month",
    "day_scaled_x_region_share_change",
    "month_cos",
    "recent_daily_mean_2",
    "blend_day_scaled_and_recent_mean_prediction",
    "trend_day_scaled_lag1_prediction",
    "month_sin",
    "log_blend_day_scaled_and_recent_mean_prediction",
    "pto_roll_mean_12",
    "settlement_number_of_cash_registers_mean",
    "number_of_households",
    "region_number_of_cash_registers_mean",
    "avg_items_per_receipt_lag_3",
    "pto_lag_1_to_region_pto_mean_lag_1",
    "trend_day_scaled_lag1_prediction_smooth",
    "log_pto_lag_1",
    "n_working_days",
    "store_share_region_change_1_2",
    "pto_roll_cv_12",
    "recent_daily_median_3",
    "households_per_cash_register",
    "day_scaled_lag2_prediction",
    "log_population_size",
    "log_daily_pto_diff_lag_1_3",
    "days_lag_2",
    "daily_growth_1_3_clipped",
    "daily_growth_1_3",
    "month_sin_2",
    "competition_per_1000_households",
    "store_share_settlement_lag2",
    "area_category_pto_lag1_mean",
]


# ============================================================
# CATBOOST LOG-LEVEL FEATURES
# ============================================================

catboost_log_level_features = [
    "log_lag1_x_region_feb_mar_growth_2024",
    "region_feb_mar_growth_2024",
    "lag1_x_global_feb_mar_growth_2024",
    "settlement_feb_mar_growth_2024",
    "lag1_x_trading_area_feb_mar_growth_2024",
    "daily_pto_lag_3",
    "settlement_poi_score_mean",
    "recent_daily_weighted_123",
    "recent_daily_mean_3_prediction",
    "log_daily_pto_lag_1",
    "day_scaled_lag1_prediction",
    "log_recent_daily_median_3_prediction",
    "log_daily_same_transition_last_year_prediction",
    "region",
    "number_of_cash_registers",
    "log_day_scaled_x_region_share_change",
    "day_scaled_x_region_share_change",
    "recent_daily_mean_3",
    "settlement_transition_prediction",
    "log_recent_daily_mean_2_prediction",
    "daily_same_transition_last_year_prediction",
    "recent_mean_anchor_to_day_scaled_ratio",
    "avg_items_per_receipt_lag_3",
    "number_of_households",
    "store_share_region_lag2",
    "blend_day_scaled_and_region_march_growth",
    "region_store_count",
    "log_daily_pto_lag_2",
    "pto_lag_1_to_settlement__trading_area_category_pto_mean_lag_1",
    "log_pto_lag_2",
    "prev_month",
    "month_name",
    "month",
    "holiday_month_type",
    "new_id",
    "population_city_type",
    "opening_date_category",
    "log_pto_lag_1",
    "log_pto_lag_3",
    "log_pto_diff_lag_1_2",
    "log_pto_diff_lag_1_3",
    "log_pto_diff_lag_3_6",
    "log_pto_diff_lag_1_12",
    "pto_lag_1_to_region_pto_mean_lag_1",
    "calendar_growth_multiplier",
    "log_calendar_heuristic_prediction",
    "log_pto_lag_1_to_calendar_heuristic",
]

### Гиперпараметры четырех моделей

In [ ]:
# ============================================================
# Best model parameters registry
# ============================================================
# Naming convention:
#   *_log_level  -> модель учится на уровне target: log1p(pto)
#   *_log_diff   -> модель учится на разности: log1p(pto) - log1p(pto_lag_1)
#
# Models:
#   1. XGBoost  – log-diff target
#   2. LightGBM – log-level target
#   3. LightGBM – log-diff target
#   4. CatBoost – log-level target
# ============================================================


# ============================================================
# 1. XGBoost – log-diff target
# Best weighted-CV model
# ============================================================

xgboost_log_diff_params = {
    "booster": "gbtree",
    "objective": "reg:absoluteerror",
    "eval_metric": "mae",
    "n_estimators": 3750,
    "learning_rate": 0.05724447518091817,
    "max_depth": 6,
    "min_child_weight": 65.3911328521089,
    "subsample": 0.5747874537857779,
    "colsample_bytree": 0.9625598201907856,
    "colsample_bylevel": 0.856900419225567,
    "colsample_bynode": 0.9822224958770875,
    "reg_lambda": 160.0,
    "reg_alpha": 0.00044247306115915205,
    "gamma": 0.35689223407647563,
    "max_bin": 256,
    "grow_policy": "depthwise",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
    "early_stopping_rounds": 100,
}

# ============================================================
# 2. LightGBM – log-level target
# Best weighted-CV model for direct log1p(pto)
# ============================================================

# lightgbm_log_level_params = {
#     "max_depth": 4,
#     "num_leaves_raw": 45,
#     "n_estimators": 7000,
#     "learning_rate": 0.045883744153168295,
#     "min_child_samples": 190,
#     "min_child_weight": 0.04151404599791484,
#     "min_split_gain": 0.2870873899457944,
#     "subsample": 0.8875505740303012,
#     "colsample_bytree": 0.6708668306863741,
#     "reg_alpha": 0.00343513451831583,
#     "reg_lambda": 74.86897961843428,
#     "max_bin": 255,
# }

lightgbm_log_level_params = {
    "max_depth": 4,
    "num_leaves": 15,  # из num_leaves_raw
    "n_estimators": 8000,
    "learning_rate": 0.04531627412595812,
    "min_child_samples": 130,
    "min_child_weight": 0.040163626155384095,
    "min_split_gain": 0.02347677898370515,
    "subsample": 0.8463795928214712,
    "colsample_bytree": 0.652567919146617,
    "reg_alpha": 0.02058508138335955,
    "reg_lambda": 68.63746198978158,
    "max_bin": 255,
}

# ============================================================
# 3. LightGBM – log-diff target
# Best weighted-CV model for log1p-diff target
# ============================================================

lightgbm_log_diff_params = {
    "max_depth": 7,
    "num_leaves": 127,
    "n_estimators": 7000,
    "learning_rate": 0.013487679469219714,
    "min_child_samples": 60,
    "min_child_weight": 0.02412331238411164,
    "subsample": 0.6069273190624236,
    "subsample_freq": 3,
    "colsample_bytree": 0.8940657969599699,
    "feature_fraction_bynode": 0.9678479119119833,
    "reg_lambda": 30.0,
    "reg_alpha": 2.594629103454418e-06,
    "min_split_gain": 0.03813044360465506,
    "max_bin": 255,
    "extra_trees": True,
    "path_smooth": 0.2513953335264824,
}

# ============================================================
# 4. CatBoost – log-level target
# Best full CatBoost model
# ============================================================

# catboost_log_level_params = {
#     "loss_function": "MAE",
#     "eval_metric": "MAE",
#     "bootstrap_type": "Bayesian",

#     "iterations": 6750,
#     "learning_rate": 0.02128577619581686,
#     "depth": 5,
#     "l2_leaf_reg": 30,
#     "random_strength": 27.36974751448375,

#     "one_hot_max_size": 42,
#     "max_ctr_complexity": 1,
#     "border_count": 192,

#     "has_time": True,
#     "leaf_estimation_iterations": 1,
#     "bagging_temperature": 0.192892090385178,

#     "grow_policy": "SymmetricTree",

#     "random_seed": 42,
#     "allow_writing_files": False,

#     "task_type": "GPU",
#     "metric_period": 100,
#     "verbose": False,
#     "devices": "0",
# }


catboost_log_level_params = {
    "loss_function": "MAE",
    "bootstrap_type": "Bayesian",
    "iterations": 6500,
    "learning_rate": 0.017054252342948984,
    "depth": 5,
    "l2_leaf_reg": 3,
    "random_strength": 21.38148662552484,
    "one_hot_max_size": 29,
    "max_ctr_complexity": 1,
    "border_count": 192,
    "has_time": True,
    "leaf_estimation_iterations": 1,
    "eval_metric": "MAE",
    "bagging_temperature": 0.15112160137554814,
    # контроль вывода
    "verbose": 500,
}

### Конфигурация признаков и вспомогательные функции подготовки данных

In [ ]:
ID_COL = "new_id"


YEAR_COL = "year"


MONTH_COL = "month"


TIME_COL = "year_month_index"


TARGET_COL = "pto"


COMMON_BASE_FEATURE_KWARGS = {
    "target_history_scale": "both",
    "monthly_history_scale": "both",
    "target_rolling_windows": (3, 6, 12),
    "monthly_cross_lags": (1, 3, 12),
    "monthly_cross_feature_types": (
        "promo_share",
        "promo_x_items",
        "items_x_hours",
        "cancellations_per_item",
        "cancellations_per_hour",
    ),
    "drop_obvious_duplicates": False,
    "fill_history_na": False,
    "downcast_float_features": True,
}


def _unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _is_cat_like_dtype(series: pd.Series) -> bool:
    dtype = series.dtype
    return (
        pd.api.types.is_object_dtype(dtype)
        or pd.api.types.is_string_dtype(dtype)
        or isinstance(dtype, pd.CategoricalDtype)
    )


def _normalize_features_for_manual_cb_encoding(selected_features):
    """
    prepare_lgbm_matrices_with_time_aware_catboost_encoding expects raw categorical
    columns as input, then creates *_cb columns.

    Example:
    - region_cb -> region
    - month_cat_cb -> month_cat, with numeric source month
    """
    if selected_features is None:
        return None, None, ()

    features_for_prepare = []
    explicit_cat_features = []
    numeric_cat_sources = []

    for f in list(selected_features):
        if not isinstance(f, str):
            continue

        if f.endswith("_cb"):
            base = f[:-3]  # remove "_cb"
            features_for_prepare.append(base)
            explicit_cat_features.append(base)

            if base.endswith("_cat"):
                numeric_cat_sources.append(base[:-4])  # remove "_cat"
        else:
            features_for_prepare.append(f)

            if f.endswith("_cat"):
                explicit_cat_features.append(f)
                numeric_cat_sources.append(f[:-4])

    features_for_prepare = _unique_keep_order(features_for_prepare)
    explicit_cat_features = _unique_keep_order(explicit_cat_features)
    numeric_cat_sources = tuple(_unique_keep_order(numeric_cat_sources))

    return features_for_prepare, explicit_cat_features, numeric_cat_sources


def _make_log_level_targets(train_df, future_df=None):
    y_train = make_forecast_target(
        train_df,
        mode="direct_log1p",
        target_col=TARGET_COL,
    )

    y_future = None
    if (
        future_df is not None
        and TARGET_COL in future_df.columns
        and future_df[TARGET_COL].notna().any()
    ):
        y_future = make_forecast_target(
            future_df,
            mode="direct_log1p",
            target_col=TARGET_COL,
        )

    return y_train, y_future


def _assert_features_available(df_, features, *, label):
    missing = [c for c in features if c not in df_.columns]
    if missing:
        raise ValueError(
            f"[{label}] Не хватает признаков: {missing}\n"
            f"Всего не хватает: {len(missing)}.\n"
            f"Пример доступных колонок: {list(df_.columns[:50])}"
        )


def _assert_no_non_numeric(df_, *, label):
    bad_cols = [c for c in df_.columns if _is_cat_like_dtype(df_[c])]
    if bad_cols:
        raise ValueError(f"[{label}] В numeric matrix остались нечисловые колонки: {bad_cols[:50]}")


def _print_dataset_shapes(title, objects):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    for name, obj in objects.items():
        if obj is None:
            print(f"{name}: None")
        elif hasattr(obj, "shape"):
            print(f"{name}: {obj.shape}")
        else:
            print(f"{name}: {type(obj)}")


def _prepare_numeric_model_pack(
    feature_output,
    *,
    selected_features,
    target_mode,
    label,
    smoothing=20.0,
    min_count=1,
    first_block_value="nan",
    unknown_value="global",
    verbose=True,
):
    """
    Готовит numeric matrices для XGBoost/LightGBM.

    Важно:
    - selected_features передаются в исходном виде, включая *_cb;
    - *_cb признаки создаются внутри prepare_lgbm_matrices...;
    - на выходе проверяем, что исходный selected_features можно взять через X[selected_features].
    """
    features_for_prepare, explicit_cat_features, numeric_cat_sources = (
        _normalize_features_for_manual_cb_encoding(selected_features)
    )

    pack = prepare_lgbm_matrices_with_time_aware_catboost_encoding(
        feature_output,
        features_to_use=features_for_prepare,
        target_mode=target_mode,
        target_col=TARGET_COL,
        id_col=ID_COL,
        time_col=TIME_COL,
        cat_features=explicit_cat_features if explicit_cat_features else None,
        include_all_detected_cat_features=False,
        add_numeric_cat_copies=bool(numeric_cat_sources),
        numeric_cat_source_cols=numeric_cat_sources,
        cb_suffix="_cb",
        smoothing=smoothing,
        min_count=min_count,
        drop_original_cat_features=True,
        drop_numeric_cat_source_cols=False,
        keep_time_col=True,
        keep_id_col=True,
        strict_train_time_order=True,
        first_block_value=first_block_value,
        unknown_value=unknown_value,
        verbose=verbose,
    )

    if pack.get("validation") is not None:
        _assert_features_available(
            pack["validation"]["X_train"],
            selected_features,
            label=f"{label}: validation X_train",
        )
        _assert_features_available(
            pack["validation"]["X_val"],
            selected_features,
            label=f"{label}: validation X_val",
        )
        _assert_no_non_numeric(
            pack["validation"]["X_train"][selected_features],
            label=f"{label}: validation X_train selected features",
        )
        _assert_no_non_numeric(
            pack["validation"]["X_val"][selected_features],
            label=f"{label}: validation X_val selected features",
        )

    if pack.get("test") is not None:
        _assert_features_available(
            pack["test"]["X_train"],
            selected_features,
            label=f"{label}: final X_train",
        )
        _assert_features_available(
            pack["test"]["X_test"],
            selected_features,
            label=f"{label}: final X_test",
        )
        _assert_no_non_numeric(
            pack["test"]["X_train"][selected_features],
            label=f"{label}: final X_train selected features",
        )
        _assert_no_non_numeric(
            pack["test"]["X_test"][selected_features],
            label=f"{label}: final X_test selected features",
        )

    if verbose:
        print(f"\n[{label}] prepared numeric model pack")
        if pack.get("validation") is not None:
            print("validation X_train:", pack["validation"]["X_train"].shape)
            print("validation X_val:  ", pack["validation"]["X_val"].shape)
        if pack.get("test") is not None:
            print("final X_train:", pack["test"]["X_train"].shape)
            print("final X_test: ", pack["test"]["X_test"].shape)

    return pack

### Функции CatBoost

In [ ]:
# ============================================================
# Compact CatBoost log-level pipeline
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostRegressor, Pool


def _cb_inverse_log1p(pred):
    pred = np.asarray(pred, dtype=float)
    out = np.expm1(pred)
    out = np.where(np.isfinite(out), out, np.nan)
    return np.maximum(out, 0.0)


def _cb_prepare_X(df, features, cat_cols, name="X"):
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Не хватает признаков: {missing}")

    X = df[features].copy()

    for c in cat_cols:
        if c in X.columns:
            X[c] = (
                X[c]
                .astype("string")
                .fillna("__missing__")
                .replace(
                    {
                        "<NA>": "__missing__",
                        "nan": "__missing__",
                        "NaN": "__missing__",
                        "None": "__missing__",
                        "none": "__missing__",
                        "NULL": "__missing__",
                        "null": "__missing__",
                    }
                )
            )

    num_cols = [c for c in X.columns if c not in cat_cols]
    if num_cols:
        X[num_cols] = X[num_cols].replace([np.inf, -np.inf], np.nan)

    return X


def _make_metrics(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    mask_mape = mask & (np.abs(y_true) > eps)

    return {
        "mae": float(np.mean(np.abs(y_pred[mask] - y_true[mask]))) if mask.any() else np.nan,
        "rmse": float(np.sqrt(np.mean((y_pred[mask] - y_true[mask]) ** 2)))
        if mask.any()
        else np.nan,
        "mape": float(
            np.mean(np.abs((y_pred[mask_mape] - y_true[mask_mape]) / y_true[mask_mape])) * 100
        )
        if mask_mape.any()
        else np.nan,
        "bias_pct": float(
            np.mean((y_pred[mask_mape] - y_true[mask_mape]) / y_true[mask_mape]) * 100
        )
        if mask_mape.any()
        else np.nan,
    }


def _make_prediction_df(
    meta_df,
    *,
    raw_pred,
    rto_pred,
    model_name,
    block,
    id_col="new_id",
    target_col="pto",
    extra_cols=None,
):
    extra_cols = [] if extra_cols is None else list(extra_cols)

    base_cols = [
        id_col,
        "year",
        "month",
        "year_month_index",
        "region",
        "settlement",
        "opening_date_category",
        "trading_area_category",
        "population_city_type",
        "lag1_rto_size_bin",
        "month_name",
        "holiday_month_type",
    ]

    keep_cols = []
    for c in base_cols + extra_cols:
        if c in meta_df.columns and c not in keep_cols:
            keep_cols.append(c)

    out = meta_df[keep_cols].reset_index(drop=True).copy()
    out["model_name"] = model_name
    out["block"] = block
    out["raw_pred"] = np.asarray(raw_pred, dtype=float)
    out["rto_pred"] = np.asarray(rto_pred, dtype=float)

    if target_col in meta_df.columns and meta_df[target_col].notna().any():
        y = meta_df[target_col].reset_index(drop=True).astype(float)
        out["rto_true"] = y
        out["signed_error"] = out["rto_pred"] - out["rto_true"]
        out["abs_error"] = out["signed_error"].abs()

        denom = out["rto_true"].replace(0, np.nan)
        out["signed_pct_error"] = 100 * out["signed_error"] / denom
        out["ape"] = 100 * out["abs_error"] / denom
        out["pred_to_true_ratio"] = out["rto_pred"] / denom
    else:
        out["rto_true"] = np.nan
        out["signed_error"] = np.nan
        out["abs_error"] = np.nan
        out["signed_pct_error"] = np.nan
        out["ape"] = np.nan
        out["pred_to_true_ratio"] = np.nan

    return out


def _make_error_aggs(pred_df, id_col="new_id", group_specs=None):
    if group_specs is None:
        group_specs = [
            id_col,
            "region",
            "settlement",
            "opening_date_category",
            "trading_area_category",
            "population_city_type",
            "lag1_rto_size_bin",
            ["region", "trading_area_category"],
            ["region", "opening_date_category"],
            ["settlement", "trading_area_category"],
        ]

    if "rto_true" not in pred_df.columns or pred_df["rto_true"].isna().all():
        return {}

    total_abs = pred_df["abs_error"].sum()
    total_true = pred_df["rto_true"].sum()
    result = {}

    for spec in group_specs:
        group_cols = [spec] if isinstance(spec, str) else list(spec)
        group_cols = [c for c in group_cols if c in pred_df.columns]
        if not group_cols:
            continue

        name = "__".join(group_cols)

        agg = (
            pred_df.groupby(group_cols, dropna=False)
            .agg(
                n_rows=(id_col, "size"),
                n_stores=(id_col, "nunique"),
                rto_true_sum=("rto_true", "sum"),
                rto_pred_sum=("rto_pred", "sum"),
                signed_error_sum=("signed_error", "sum"),
                abs_error_sum=("abs_error", "sum"),
                mae=("abs_error", "mean"),
                mape=("ape", "mean"),
                median_ape=("ape", "median"),
                p90_ape=(
                    "ape",
                    lambda x: np.nanpercentile(x.dropna(), 90) if x.notna().any() else np.nan,
                ),
                bias_pct=("signed_pct_error", "mean"),
            )
            .reset_index()
        )

        agg["abs_error_share"] = agg["abs_error_sum"] / total_abs if total_abs else np.nan
        agg["true_rto_share"] = agg["rto_true_sum"] / total_true if total_true else np.nan
        agg["sum_bias_pct"] = 100 * agg["signed_error_sum"] / agg["rto_true_sum"].replace(0, np.nan)

        result[name] = agg.sort_values(
            ["abs_error_share", "abs_error_sum"],
            ascending=False,
        ).reset_index(drop=True)

    return result


def run_catboost_log_level(
    *,
    params,
    features,
    val_train_df,
    val_test_df,
    val_y_train,
    val_y_test=None,
    full_train_df,
    full_test_df,
    full_y_train=None,
    cat_cols=None,
    numeric_cat_cols=("new_id", "month", "prev_month"),
    id_col="new_id",
    target_col="pto",
    model_name="catboost_log_level",
    output_dir="outputs_catboost_log_level",
    submission_filename="test_catboost_log_level.csv",
    early_stopping_rounds=100,
    use_best_model=True,
    final_iterations_strategy="params",  # "params" or "best_iteration"
    save_files=True,
    verbose=True,
):
    features = list(features)

    # ------------------------------------------------------------
    # Cat columns
    # ------------------------------------------------------------
    auto_cat_cols = [
        c
        for c in features
        if c in val_train_df.columns
        and (
            pd.api.types.is_object_dtype(val_train_df[c].dtype)
            or pd.api.types.is_string_dtype(val_train_df[c].dtype)
            or isinstance(val_train_df[c].dtype, pd.CategoricalDtype)
        )
    ]

    cat_cols = [] if cat_cols is None else list(cat_cols)
    numeric_cat_cols = [] if numeric_cat_cols is None else list(numeric_cat_cols)

    cat_features = []
    for c in auto_cat_cols + cat_cols + numeric_cat_cols:
        if c in features and c in val_train_df.columns and c not in cat_features:
            cat_features.append(c)

    # ------------------------------------------------------------
    # Validation fit
    # ------------------------------------------------------------
    X_tr = _cb_prepare_X(val_train_df, features, cat_features, name="catboost_val_train")
    X_va = _cb_prepare_X(val_test_df, features, cat_features, name="catboost_val_test")

    y_tr = pd.Series(val_y_train).astype(float).reset_index(drop=True)
    if len(X_tr) != len(y_tr):
        raise ValueError(f"len(X_tr) != len(y_tr): {len(X_tr)} != {len(y_tr)}")

    fit_params = {}
    if val_y_test is not None:
        y_va = pd.Series(val_y_test).astype(float).reset_index(drop=True)
        fit_params["eval_set"] = Pool(X_va, y_va, cat_features=cat_features)
        fit_params["use_best_model"] = bool(use_best_model)

        if early_stopping_rounds is not None:
            fit_params["early_stopping_rounds"] = int(early_stopping_rounds)

    clean_params = dict(params)
    clean_params.pop("early_stopping_rounds", None)
    clean_params.pop("use_best_model", None)

    val_model = CatBoostRegressor(**clean_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] validation fit")
        print("=" * 80)
        print("X_train:", X_tr.shape)
        print("X_val:  ", X_va.shape)
        print("features:", len(features))
        print("cat_features:", len(cat_features), cat_features)

    val_model.fit(
        Pool(X_tr, y_tr, cat_features=cat_features),
        **fit_params,
    )

    raw_val_pred = val_model.predict(X_va)
    val_pred = _cb_inverse_log1p(raw_val_pred)

    val_true = (
        val_test_df[target_col].to_numpy(dtype=float)
        if target_col in val_test_df.columns and val_test_df[target_col].notna().any()
        else None
    )

    val_predictions_df = _make_prediction_df(
        val_test_df,
        raw_pred=raw_val_pred,
        rto_pred=val_pred,
        model_name=model_name,
        block="validation",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    val_metrics = {
        "model_name": model_name,
        "n_val_rows": len(val_predictions_df),
        **(_make_metrics(val_true, val_pred) if val_true is not None else {}),
        "best_iteration": val_model.get_best_iteration(),
        "tree_count": int(val_model.tree_count_),
    }

    error_aggs = _make_error_aggs(val_predictions_df, id_col=id_col)

    if verbose:
        print("\nValidation metrics:")
        for k, v in val_metrics.items():
            print(f"{k}: {v}")

    # ------------------------------------------------------------
    # Full fit
    # ------------------------------------------------------------
    if full_y_train is None:
        full_y_train = np.log1p(full_train_df[target_col].clip(lower=0))

    X_full = _cb_prepare_X(full_train_df, features, cat_features, name="catboost_full_train")
    X_test = _cb_prepare_X(full_test_df, features, cat_features, name="catboost_full_test")

    y_full = pd.Series(full_y_train).astype(float).reset_index(drop=True)
    if len(X_full) != len(y_full):
        raise ValueError(f"len(X_full) != len(y_full): {len(X_full)} != {len(y_full)}")

    final_params = dict(clean_params)

    if final_iterations_strategy == "best_iteration":
        best_iter = val_model.get_best_iteration()
        if best_iter is not None and best_iter > 0:
            final_params["iterations"] = int(best_iter) + 1
    elif final_iterations_strategy != "params":
        raise ValueError("final_iterations_strategy должен быть 'params' или 'best_iteration'.")

    full_model = CatBoostRegressor(**final_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] full fit")
        print("=" * 80)
        print("X_full:", X_full.shape)
        print("X_test:", X_test.shape)

    full_model.fit(Pool(X_full, y_full, cat_features=cat_features))

    raw_test_pred = full_model.predict(X_test)
    test_pred = _cb_inverse_log1p(raw_test_pred)

    full_predictions_df = _make_prediction_df(
        full_test_df,
        raw_pred=raw_test_pred,
        rto_pred=test_pred,
        model_name=model_name,
        block="final_test",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    submission_df = pd.DataFrame(
        {
            id_col: full_test_df[id_col].to_numpy(),
            "rto": test_pred,
        }
    )

    submission_df["rto"] = (
        pd.to_numeric(submission_df["rto"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(lower=0)
    )

    if submission_df["rto"].isna().any():
        raise ValueError(f"[{model_name}] В submission есть NaN.")
    if submission_df[id_col].duplicated().any():
        raise ValueError(f"[{model_name}] В submission есть дубли по {id_col}.")

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    saved_paths = {}
    output_dir = Path(output_dir)

    if save_files:
        output_dir.mkdir(parents=True, exist_ok=True)

        paths = {
            "val_predictions": output_dir / f"{model_name}_validation_predictions.csv",
            "full_predictions": output_dir / f"{model_name}_final_predictions.csv",
            "metrics": output_dir / f"{model_name}_validation_metrics.csv",
            "submission": output_dir / submission_filename,
        }

        val_predictions_df.to_csv(paths["val_predictions"], index=False)
        full_predictions_df.to_csv(paths["full_predictions"], index=False)
        pd.DataFrame([val_metrics]).to_csv(paths["metrics"], index=False)
        submission_df.to_csv(paths["submission"], index=False)

        for name, agg in error_aggs.items():
            path = output_dir / f"{model_name}_errors_by_{name}.csv"
            agg.to_csv(path, index=False)
            paths[f"errors_by_{name}"] = path

        saved_paths = {k: str(v) for k, v in paths.items()}

    gc.collect()

    return {
        "model_name": model_name,
        "features": features,
        "cat_features": cat_features,
        "val_model": val_model,
        "full_model": full_model,
        "val_predictions_df": val_predictions_df,
        "full_predictions_df": full_predictions_df,
        "submission_df": submission_df,
        "val_metrics": val_metrics,
        "error_aggs": error_aggs,
        "saved_paths": saved_paths,
    }

### Функции LightGBM

In [ ]:
# ============================================================
# Compact LightGBM pipeline for log-diff / log-level targets
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path

import lightgbm as lgb
from lightgbm import LGBMRegressor


def _clean_lgbm_params(params):
    params = dict(params)

    # В твоём registry для log-level есть num_leaves_raw.
    if "num_leaves_raw" in params and "num_leaves" not in params:
        params["num_leaves"] = params.pop("num_leaves_raw")
    else:
        params.pop("num_leaves_raw", None)

    # Безопасные дефолты.
    params.setdefault("objective", "regression_l1")
    params.setdefault("metric", "mae")
    params.setdefault("random_state", 42)
    params.setdefault("n_jobs", -1)
    params.setdefault("verbosity", -1)

    # Эти параметры лучше задавать через callbacks/fit, если понадобятся.
    params.pop("early_stopping_rounds", None)

    return params


def _lgbm_predict_to_rto(raw_pred, raw_features_df, target_mode):
    return inverse_forecast_target(
        raw_prediction=raw_pred,
        features=raw_features_df,
        mode=target_mode,
        target_col="pto",
        clip_min=0.0,
    )


def run_lightgbm_pipeline(
    *,
    params,
    features,
    target_mode,  # "growth_log" or "direct_log1p"
    # numeric model matrices
    val_train_df,
    val_test_df,
    val_y_train,
    val_y_test,
    full_train_df,
    full_test_df,
    full_y_train,
    # raw/meta frames for inverse transform and diagnostics
    val_test_meta_df,
    full_test_meta_df,
    id_col="new_id",
    target_col="pto",
    model_name="lightgbm_model",
    output_dir="outputs_lightgbm",
    submission_filename="test_lightgbm.csv",
    early_stopping_rounds=100,
    final_iterations_strategy="params",  # "params" or "best_iteration"
    save_files=True,
    verbose=True,
):
    """
    Единый LightGBM pipeline:
    - обучает validation-модель;
    - прогнозирует validation;
    - считает метрики и error aggregates;
    - обучает full-модель;
    - прогнозирует final test;
    - сохраняет prediction df и submission.
    """

    if target_mode not in {"growth_log", "direct_log1p"}:
        raise ValueError("target_mode должен быть 'growth_log' или 'direct_log1p'.")

    features = list(features)

    missing_train = [c for c in features if c not in val_train_df.columns]
    missing_val = [c for c in features if c not in val_test_df.columns]
    missing_full = [c for c in features if c not in full_train_df.columns]
    missing_test = [c for c in features if c not in full_test_df.columns]

    if missing_train or missing_val or missing_full or missing_test:
        raise ValueError(
            "Не хватает признаков:\n"
            f"val_train: {missing_train}\n"
            f"val_test: {missing_val}\n"
            f"full_train: {missing_full}\n"
            f"full_test: {missing_test}"
        )

    X_tr = val_train_df[features].copy()
    X_va = val_test_df[features].copy()
    y_tr = pd.Series(val_y_train).astype(float).reset_index(drop=True)
    y_va = pd.Series(val_y_test).astype(float).reset_index(drop=True)

    X_full = full_train_df[features].copy()
    X_test = full_test_df[features].copy()
    y_full = pd.Series(full_y_train).astype(float).reset_index(drop=True)

    if len(X_tr) != len(y_tr):
        raise ValueError(f"len(X_tr) != len(y_tr): {len(X_tr)} != {len(y_tr)}")
    if len(X_va) != len(y_va):
        raise ValueError(f"len(X_va) != len(y_va): {len(X_va)} != {len(y_va)}")
    if len(X_full) != len(y_full):
        raise ValueError(f"len(X_full) != len(y_full): {len(X_full)} != {len(y_full)}")
    if len(X_va) != len(val_test_meta_df):
        raise ValueError("X_va и val_test_meta_df имеют разную длину.")
    if len(X_test) != len(full_test_meta_df):
        raise ValueError("X_test и full_test_meta_df имеют разную длину.")

    clean_params = _clean_lgbm_params(params)

    callbacks = []
    if early_stopping_rounds is not None:
        callbacks.append(
            lgb.early_stopping(
                stopping_rounds=int(early_stopping_rounds),
                verbose=bool(verbose),
            )
        )
    callbacks.append(lgb.log_evaluation(period=100 if verbose else 0))

    # ------------------------------------------------------------
    # Validation fit
    # ------------------------------------------------------------
    val_model = LGBMRegressor(**clean_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] validation fit")
        print("=" * 80)
        print("target_mode:", target_mode)
        print("X_train:", X_tr.shape)
        print("X_val:  ", X_va.shape)
        print("features:", len(features))

    val_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="mae",
        callbacks=callbacks,
    )

    best_iter = getattr(val_model, "best_iteration_", None)
    pred_iter = best_iter if best_iter and best_iter > 0 else None

    raw_val_pred = val_model.predict(X_va, num_iteration=pred_iter)
    val_pred = _lgbm_predict_to_rto(
        raw_val_pred,
        val_test_meta_df,
        target_mode=target_mode,
    )

    val_true = (
        val_test_meta_df[target_col].to_numpy(dtype=float)
        if target_col in val_test_meta_df.columns and val_test_meta_df[target_col].notna().any()
        else None
    )

    val_predictions_df = _make_prediction_df(
        val_test_meta_df,
        raw_pred=raw_val_pred,
        rto_pred=val_pred,
        model_name=model_name,
        block="validation",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    val_metrics = {
        "model_name": model_name,
        "target_mode": target_mode,
        "n_val_rows": len(val_predictions_df),
        **(_make_metrics(val_true, val_pred) if val_true is not None else {}),
        "best_iteration": int(best_iter) if best_iter else None,
        "n_estimators_param": clean_params.get("n_estimators"),
    }

    error_aggs = _make_error_aggs(val_predictions_df, id_col=id_col)

    if verbose:
        print("\nValidation metrics:")
        for k, v in val_metrics.items():
            print(f"{k}: {v}")

    # ------------------------------------------------------------
    # Full fit
    # ------------------------------------------------------------
    final_params = dict(clean_params)

    if final_iterations_strategy == "best_iteration":
        if best_iter is not None and best_iter > 0:
            final_params["n_estimators"] = int(best_iter)
    elif final_iterations_strategy != "params":
        raise ValueError("final_iterations_strategy должен быть 'params' или 'best_iteration'.")

    full_model = LGBMRegressor(**final_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] full fit")
        print("=" * 80)
        print("X_full:", X_full.shape)
        print("X_test:", X_test.shape)
        print("n_estimators:", final_params.get("n_estimators"))

    full_model.fit(X_full, y_full)

    raw_test_pred = full_model.predict(X_test)
    test_pred = _lgbm_predict_to_rto(
        raw_test_pred,
        full_test_meta_df,
        target_mode=target_mode,
    )

    full_predictions_df = _make_prediction_df(
        full_test_meta_df,
        raw_pred=raw_test_pred,
        rto_pred=test_pred,
        model_name=model_name,
        block="final_test",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    submission_df = pd.DataFrame(
        {
            id_col: full_test_meta_df[id_col].to_numpy(),
            "rto": test_pred,
        }
    )

    submission_df["rto"] = (
        pd.to_numeric(submission_df["rto"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(lower=0)
    )

    if submission_df["rto"].isna().any():
        raise ValueError(f"[{model_name}] В submission есть NaN.")
    if submission_df[id_col].duplicated().any():
        raise ValueError(f"[{model_name}] В submission есть дубли по {id_col}.")

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    saved_paths = {}
    output_dir = Path(output_dir)

    if save_files:
        output_dir.mkdir(parents=True, exist_ok=True)

        paths = {
            "val_predictions": output_dir / f"{model_name}_validation_predictions.csv",
            "full_predictions": output_dir / f"{model_name}_final_predictions.csv",
            "metrics": output_dir / f"{model_name}_validation_metrics.csv",
            "submission": output_dir / submission_filename,
        }

        val_predictions_df.to_csv(paths["val_predictions"], index=False)
        full_predictions_df.to_csv(paths["full_predictions"], index=False)
        pd.DataFrame([val_metrics]).to_csv(paths["metrics"], index=False)
        submission_df.to_csv(paths["submission"], index=False)

        for name, agg in error_aggs.items():
            path = output_dir / f"{model_name}_errors_by_{name}.csv"
            agg.to_csv(path, index=False)
            paths[f"errors_by_{name}"] = path

        saved_paths = {k: str(v) for k, v in paths.items()}

    gc.collect()

    return {
        "model_name": model_name,
        "target_mode": target_mode,
        "features": features,
        "val_model": val_model,
        "full_model": full_model,
        "val_predictions_df": val_predictions_df,
        "full_predictions_df": full_predictions_df,
        "submission_df": submission_df,
        "val_metrics": val_metrics,
        "error_aggs": error_aggs,
        "saved_paths": saved_paths,
    }

### Функции XGBoost

In [ ]:
# ============================================================
# Compact XGBoost log-diff pipeline – fixed finite target version
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor


def _xgb_prepare_X(df, features, *, name="X"):
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Не хватает признаков: {missing}")

    X = df[features].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    return X.reset_index(drop=True)


def _xgb_filter_finite_y(X, y, *, name):
    X = X.reset_index(drop=True)
    y = pd.Series(y).astype(float).reset_index(drop=True)

    if len(X) != len(y):
        raise ValueError(f"[{name}] len(X) != len(y): {len(X)} != {len(y)}")

    y_values = y.to_numpy(dtype=float)
    mask = np.isfinite(y_values)

    n_drop = int((~mask).sum())
    if n_drop > 0:
        print(f"[{name}] dropped rows with non-finite y: {n_drop}")

    X_clean = X.loc[mask].reset_index(drop=True)
    y_clean = y.loc[mask].reset_index(drop=True)

    if len(y_clean) == 0:
        raise ValueError(f"[{name}] После фильтрации finite y не осталось строк.")

    return X_clean, y_clean


def _xgb_metrics(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    mask_mape = mask & (np.abs(y_true) > eps)

    return {
        "mae": float(np.mean(np.abs(y_pred[mask] - y_true[mask]))) if mask.any() else np.nan,
        "rmse": float(np.sqrt(np.mean((y_pred[mask] - y_true[mask]) ** 2)))
        if mask.any()
        else np.nan,
        "mape": float(
            np.mean(np.abs((y_pred[mask_mape] - y_true[mask_mape]) / y_true[mask_mape])) * 100
        )
        if mask_mape.any()
        else np.nan,
        "bias_pct": float(
            np.mean((y_pred[mask_mape] - y_true[mask_mape]) / y_true[mask_mape]) * 100
        )
        if mask_mape.any()
        else np.nan,
    }


def _xgb_make_prediction_df(
    meta_df,
    *,
    raw_pred,
    rto_pred,
    model_name,
    block,
    id_col="new_id",
    target_col="pto",
    extra_cols=None,
):
    extra_cols = [] if extra_cols is None else list(extra_cols)

    base_cols = [
        id_col,
        "year",
        "month",
        "year_month_index",
        "region",
        "settlement",
        "opening_date_category",
        "trading_area_category",
        "population_city_type",
        "lag1_rto_size_bin",
        "month_name",
        "holiday_month_type",
    ]

    keep_cols = []
    for c in base_cols + extra_cols:
        if c in meta_df.columns and c not in keep_cols:
            keep_cols.append(c)

    out = meta_df[keep_cols].reset_index(drop=True).copy()
    out["model_name"] = model_name
    out["block"] = block
    out["raw_pred"] = np.asarray(raw_pred, dtype=float)
    out["rto_pred"] = np.asarray(rto_pred, dtype=float)

    if target_col in meta_df.columns and meta_df[target_col].notna().any():
        out["rto_true"] = meta_df[target_col].reset_index(drop=True).astype(float)
        out["signed_error"] = out["rto_pred"] - out["rto_true"]
        out["abs_error"] = out["signed_error"].abs()

        denom = out["rto_true"].replace(0, np.nan)
        out["signed_pct_error"] = 100 * out["signed_error"] / denom
        out["ape"] = 100 * out["abs_error"] / denom
        out["pred_to_true_ratio"] = out["rto_pred"] / denom
    else:
        out["rto_true"] = np.nan
        out["signed_error"] = np.nan
        out["abs_error"] = np.nan
        out["signed_pct_error"] = np.nan
        out["ape"] = np.nan
        out["pred_to_true_ratio"] = np.nan

    return out


def _xgb_error_aggs(pred_df, id_col="new_id", group_specs=None):
    if group_specs is None:
        group_specs = [
            id_col,
            "region",
            "settlement",
            "opening_date_category",
            "trading_area_category",
            "population_city_type",
            "lag1_rto_size_bin",
            ["region", "trading_area_category"],
            ["region", "opening_date_category"],
            ["settlement", "trading_area_category"],
        ]

    if "rto_true" not in pred_df.columns or pred_df["rto_true"].isna().all():
        return {}

    total_abs = pred_df["abs_error"].sum()
    total_true = pred_df["rto_true"].sum()
    result = {}

    for spec in group_specs:
        group_cols = [spec] if isinstance(spec, str) else list(spec)
        group_cols = [c for c in group_cols if c in pred_df.columns]
        if not group_cols:
            continue

        name = "__".join(group_cols)

        agg = (
            pred_df.groupby(group_cols, dropna=False)
            .agg(
                n_rows=(id_col, "size"),
                n_stores=(id_col, "nunique"),
                rto_true_sum=("rto_true", "sum"),
                rto_pred_sum=("rto_pred", "sum"),
                signed_error_sum=("signed_error", "sum"),
                abs_error_sum=("abs_error", "sum"),
                mae=("abs_error", "mean"),
                mape=("ape", "mean"),
                median_ape=("ape", "median"),
                p90_ape=(
                    "ape",
                    lambda x: np.nanpercentile(x.dropna(), 90) if x.notna().any() else np.nan,
                ),
                bias_pct=("signed_pct_error", "mean"),
            )
            .reset_index()
        )

        agg["abs_error_share"] = agg["abs_error_sum"] / total_abs if total_abs else np.nan
        agg["true_rto_share"] = agg["rto_true_sum"] / total_true if total_true else np.nan
        agg["sum_bias_pct"] = 100 * agg["signed_error_sum"] / agg["rto_true_sum"].replace(0, np.nan)

        result[name] = agg.sort_values(
            ["abs_error_share", "abs_error_sum"],
            ascending=False,
        ).reset_index(drop=True)

    return result


def _clean_xgb_params(params, *, keep_early_stopping):
    params = dict(params)

    early_stopping_rounds = params.pop("early_stopping_rounds", None)

    if keep_early_stopping and early_stopping_rounds is not None:
        params["early_stopping_rounds"] = int(early_stopping_rounds)

    params.setdefault("objective", "reg:absoluteerror")
    params.setdefault("eval_metric", "mae")
    params.setdefault("random_state", 42)
    params.setdefault("n_jobs", -1)

    return params


def run_xgboost_log_diff(
    *,
    params,
    features,
    val_train_df,
    val_test_df,
    val_y_train,
    val_y_test,
    full_train_df,
    full_test_df,
    full_y_train,
    val_test_meta_df,
    full_test_meta_df,
    id_col="new_id",
    target_col="pto",
    model_name="xgboost_log_diff",
    output_dir="outputs_xgboost_log_diff",
    submission_filename="test_xgboost_log_diff.csv",
    final_iterations_strategy="params",  # "params" or "best_iteration"
    save_files=True,
    verbose=True,
):
    features = list(features)

    # ------------------------------------------------------------
    # Prepare matrices
    # ------------------------------------------------------------
    X_tr_raw = _xgb_prepare_X(val_train_df, features, name=f"{model_name}:val_train_df")
    X_va_full = _xgb_prepare_X(val_test_df, features, name=f"{model_name}:val_test_df")

    X_full_raw = _xgb_prepare_X(full_train_df, features, name=f"{model_name}:full_train_df")
    X_test = _xgb_prepare_X(full_test_df, features, name=f"{model_name}:full_test_df")

    if len(X_va_full) != len(val_test_meta_df):
        raise ValueError(
            f"X_va_full и val_test_meta_df имеют разную длину: "
            f"{len(X_va_full)} != {len(val_test_meta_df)}"
        )

    if len(X_test) != len(full_test_meta_df):
        raise ValueError(
            f"X_test и full_test_meta_df имеют разную длину: "
            f"{len(X_test)} != {len(full_test_meta_df)}"
        )

    # XGBoost не принимает NaN/inf в label.
    # Поэтому fit/eval обучаем только на строках с finite y.
    X_tr, y_tr = _xgb_filter_finite_y(
        X_tr_raw,
        val_y_train,
        name=f"{model_name}: validation train",
    )

    X_va_fit, y_va_fit = _xgb_filter_finite_y(
        X_va_full,
        val_y_test,
        name=f"{model_name}: validation eval",
    )

    X_full, y_full = _xgb_filter_finite_y(
        X_full_raw,
        full_y_train,
        name=f"{model_name}: full train",
    )

    # ------------------------------------------------------------
    # Validation fit
    # ------------------------------------------------------------
    val_params = _clean_xgb_params(params, keep_early_stopping=True)
    val_model = XGBRegressor(**val_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] validation fit")
        print("=" * 80)
        print("X_train raw:", X_tr_raw.shape)
        print("X_train fit:", X_tr.shape)
        print("X_val full: ", X_va_full.shape)
        print("X_val eval: ", X_va_fit.shape)
        print("features:", len(features))
        print("n_estimators:", val_params.get("n_estimators"))
        print("early_stopping_rounds:", val_params.get("early_stopping_rounds"))

    val_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va_fit, y_va_fit)],
        verbose=100 if verbose else False,
    )

    best_iteration = getattr(val_model, "best_iteration", None)

    # Предсказываем на полном validation, чтобы сохранить все магазины.
    raw_val_pred = val_model.predict(X_va_full)

    val_pred = inverse_forecast_target(
        raw_prediction=raw_val_pred,
        features=val_test_meta_df,
        mode="growth_log",
        target_col=target_col,
        clip_min=0.0,
    )

    val_true = (
        val_test_meta_df[target_col].to_numpy(dtype=float)
        if target_col in val_test_meta_df.columns and val_test_meta_df[target_col].notna().any()
        else None
    )

    val_predictions_df = _xgb_make_prediction_df(
        val_test_meta_df,
        raw_pred=raw_val_pred,
        rto_pred=val_pred,
        model_name=model_name,
        block="validation",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    val_metrics = {
        "model_name": model_name,
        "target_mode": "growth_log",
        "n_val_rows": len(val_predictions_df),
        **(_xgb_metrics(val_true, val_pred) if val_true is not None else {}),
        "best_iteration": int(best_iteration) if best_iteration is not None else None,
        "n_estimators_param": val_params.get("n_estimators"),
        "n_train_raw": len(X_tr_raw),
        "n_train_fit": len(X_tr),
        "n_eval_raw": len(X_va_full),
        "n_eval_fit": len(X_va_fit),
    }

    error_aggs = _xgb_error_aggs(val_predictions_df, id_col=id_col)

    if verbose:
        print("\nValidation metrics:")
        for k, v in val_metrics.items():
            print(f"{k}: {v}")

    # ------------------------------------------------------------
    # Full fit
    # ------------------------------------------------------------
    final_params = _clean_xgb_params(params, keep_early_stopping=False)

    if final_iterations_strategy == "best_iteration":
        if best_iteration is not None and best_iteration > 0:
            final_params["n_estimators"] = int(best_iteration) + 1
    elif final_iterations_strategy != "params":
        raise ValueError("final_iterations_strategy должен быть 'params' или 'best_iteration'.")

    full_model = XGBRegressor(**final_params)

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{model_name}] full fit")
        print("=" * 80)
        print("X_full raw:", X_full_raw.shape)
        print("X_full fit:", X_full.shape)
        print("X_test:    ", X_test.shape)
        print("n_estimators:", final_params.get("n_estimators"))

    full_model.fit(
        X_full,
        y_full,
        verbose=100 if verbose else False,
    )

    raw_test_pred = full_model.predict(X_test)

    test_pred = inverse_forecast_target(
        raw_prediction=raw_test_pred,
        features=full_test_meta_df,
        mode="growth_log",
        target_col=target_col,
        clip_min=0.0,
    )

    full_predictions_df = _xgb_make_prediction_df(
        full_test_meta_df,
        raw_pred=raw_test_pred,
        rto_pred=test_pred,
        model_name=model_name,
        block="final_test",
        id_col=id_col,
        target_col=target_col,
        extra_cols=[
            "pto_lag_1",
            "day_scaled_lag1_prediction",
            "calendar_heuristic_prediction",
            "recent_daily_mean_3_prediction",
            "daily_same_transition_last_year_prediction",
            "region_feb_mar_growth_2024",
            "settlement_feb_mar_growth_2024",
        ],
    )

    submission_df = pd.DataFrame(
        {
            id_col: full_test_meta_df[id_col].to_numpy(),
            "rto": test_pred,
        }
    )

    submission_df["rto"] = (
        pd.to_numeric(submission_df["rto"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(lower=0)
    )

    if submission_df["rto"].isna().any():
        raise ValueError(f"[{model_name}] В submission есть NaN.")
    if submission_df[id_col].duplicated().any():
        raise ValueError(f"[{model_name}] В submission есть дубли по {id_col}.")

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    saved_paths = {}
    output_dir = Path(output_dir)

    if save_files:
        output_dir.mkdir(parents=True, exist_ok=True)

        paths = {
            "val_predictions": output_dir / f"{model_name}_validation_predictions.csv",
            "full_predictions": output_dir / f"{model_name}_final_predictions.csv",
            "metrics": output_dir / f"{model_name}_validation_metrics.csv",
            "submission": output_dir / submission_filename,
        }

        val_predictions_df.to_csv(paths["val_predictions"], index=False)
        full_predictions_df.to_csv(paths["full_predictions"], index=False)
        pd.DataFrame([val_metrics]).to_csv(paths["metrics"], index=False)
        submission_df.to_csv(paths["submission"], index=False)

        for name, agg in error_aggs.items():
            path = output_dir / f"{model_name}_errors_by_{name}.csv"
            agg.to_csv(path, index=False)
            paths[f"errors_by_{name}"] = path

        saved_paths = {k: str(v) for k, v in paths.items()}

    gc.collect()

    return {
        "model_name": model_name,
        "target_mode": "growth_log",
        "features": features,
        "val_model": val_model,
        "full_model": full_model,
        "val_predictions_df": val_predictions_df,
        "full_predictions_df": full_predictions_df,
        "submission_df": submission_df,
        "val_metrics": val_metrics,
        "error_aggs": error_aggs,
        "saved_paths": saved_paths,
    }

## Данные и разбиение для прогноза на март 2025 года

Признаки строятся исходной функцией по всей истории январь 2023 - февраль 2025 года, строки марта
создаются из последнего известного месяца. Фактическая мартовская выручка не используется.

In [ ]:
raw = pd.read_csv(SHOPS_PATH).rename(columns=rename_cols)
assert not raw.duplicated(["new_id", "year", "month"]).any() and raw.pto.notna().all()

features = build_retail_features_with_split(
    raw, split_mode="train_val_test", min_year=2023, val_year=2025, val_month=2,
    test_year=2025, test_month=3, create_test_if_missing=True, expected_test_size=18657,
    use_val_as_test_history=True, base_feature_kwargs=COMMON_BASE_FEATURE_KWARGS, verbose=True,
)

# Компактный набор: только признаки четырех моделей и служебные поля
requested = {"new_id", "year_month_index", "pto", "pto_lag_1", "region", "settlement"}
for model in MODELS:
    for name in globals()[model + "_features"]:
        requested.add(name[:-3] if name.endswith("_cb") else name)
        if name.endswith("_cat_cb"):
            requested.add(name[:-7])
compact = {}
for key in ["test_train_features", "test_features"]:
    frame = features[key]
    compact[key] = frame[[c for c in frame.columns if c in requested]].copy()
compact["categorical_cols"] = features["categorical_cols"]
compact["feature_cols"] = features["feature_cols"]
del features
gc.collect()

assert len(compact["test_train_features"]) == 485082 and len(compact["test_features"]) == 18657
assert compact["test_train_features"].year_month_index.max() == 25
assert compact["test_features"].pto.isna().all()
# Обучение на всей истории, прогноз на март
forecast_data = dict(compact, train_features=compact["test_train_features"],
                     val_features=compact["test_features"])

## Привязка магазинов к населенным пунктам

Магазин относится к населенному пункту по нормализованным названиям субъекта, населенного пункта и
типа населенного пункта. Координаты берутся из базы GeoNames и уточняются по итоговой панели
основного исследования.

In [ ]:
import re
import unicodedata

LATIN_LOOKALIKES = str.maketrans("ABCEHKMOPTXaceopxy", "АВСЕНКМОРТХасеорху")
REGION_WORD_EXPANSIONS = {"обл": "область", "респ": "республика", "авт": "автономный",
                          "автономная": "автономный", "ао": "автономный округ", "г": "город"}
REGION_OPTIONAL_WORDS = {"кузбасс", "чувашия"}
SETTLEMENT_TYPE_CLASSES = {
    "г": "город", "город": "город", "пгт": "поселок городского типа", "рп": "поселок городского типа",
    "гп": "поселок городского типа", "дп": "поселок городского типа", "кп": "поселок городского типа",
    "п": "поселок", "пос": "поселок", "поселок": "поселок", "с": "село", "д": "деревня", "дер": "деревня",
    "ст-ца": "станица", "х": "хутор", "аул": "аул", "сл": "слобода", "ст": "станция", "ж/д_ст": "станция",
    "п/ст": "станция", "м": "местечко", "нп": "населенный пункт", "сп": "сельский поселок",
    "г-к": "городок", "р-н": "район",
}


def clean_text(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(LATIN_LOOKALIKES)
    return text.lower().replace("\u0451", "е").replace("\n", " ").strip()


def region_key(region_name):
    """Ключ субъекта - отсортированные слова названия без уточнений."""
    text = clean_text(region_name).replace("в том числе", " ")
    without_autonomous_okrugs = bool(re.search(r"\bкроме\b|\bбез\b", text))
    text = re.split(r"\(|\bбез\b|\bвключая\b|,", text)[0]
    words = set()
    for word in re.findall(r"[а-яa-z0-9]+", text):
        words.update(REGION_WORD_EXPANSIONS.get(word, word).split())
    words -= REGION_OPTIONAL_WORDS
    key = " ".join(sorted(words))
    return f"{key} без_ао" if without_autonomous_okrugs else key


def split_settlement(raw_name):
    """Разделяет запись населенного пункта на название и тип."""
    words = clean_text(raw_name).replace("населенный пункт", "нп").split()
    types = [word.rstrip(".") for word in words]
    if len(words) > 1 and types[-1] in SETTLEMENT_TYPE_CLASSES:
        name_words, settlement_type = words[:-1], types[-1]
    elif len(words) > 1 and types[0] in SETTLEMENT_TYPE_CLASSES:
        name_words, settlement_type = words[1:], types[0]
    else:
        raise ValueError(f"Не распознан тип населенного пункта: {raw_name!r}")
    if len(name_words) > 1 and name_words[0].rstrip(".") in SETTLEMENT_TYPE_CLASSES:
        name_words = name_words[1:]
    return " ".join(name_words), settlement_type


def settlement_base_name(name):
    text = re.sub(r"\([^)]*\)", " ", clean_text(name))
    return re.sub(r"[^а-яa-z0-9]+", "", text)


def settlement_type_class(settlement_type):
    return SETTLEMENT_TYPE_CLASSES[settlement_type.rstrip(".").lower()]



def location_key(region, settlement):
    try:
        name, settlement_type = split_settlement(settlement)
        return region_key(region) + "|" + settlement_base_name(name) + "|" + settlement_type_class(settlement_type)
    except ValueError:
        return region_key(region) + "|unparsed|" + str(settlement).strip().lower()


stores = raw[["new_id", "region", "settlement"]].drop_duplicates().copy()
assert not stores.new_id.duplicated().any()
stores["location_key"] = [location_key(r, s) for r, s in zip(stores.region, stores.settlement)]

coordinates = pd.read_csv(COORDINATES_PATH, usecols=["region", "settlement", "settlement_lat", "settlement_lon"])
coordinates["location_key"] = [location_key(r, s) for r, s in zip(coordinates.region, coordinates.settlement)]
unique = coordinates[["location_key", "settlement_lat", "settlement_lon"]].drop_duplicates()
unique = unique.loc[~unique.location_key.duplicated(keep=False)].set_index("location_key")

panel = pd.read_parquet(SETTLEMENT_PANEL_PATH, columns=["shop_region", "settlement_name",
                        "settlement_type_class", "latitude", "longitude"]).drop_duplicates()
panel["location_key"] = [region_key(r) + "|" + settlement_base_name(n) + "|" + t for r, n, t in
                         zip(panel.shop_region, panel.settlement_name, panel.settlement_type_class)]
overrides = panel.set_index("location_key")[["latitude", "longitude"]].rename(
    columns={"latitude": "settlement_lat", "longitude": "settlement_lon"})
unique = pd.concat([unique.loc[~unique.index.isin(overrides.index)], overrides])

locations = (stores[["location_key", "region", "settlement"]].drop_duplicates("location_key")
             .sort_values("location_key").join(unique, on="location_key").reset_index(drop=True))
locations["node"] = np.arange(len(locations))
stores = stores.merge(locations[["location_key", "node"]], on="location_key", validate="many_to_one")
store_nodes = stores.set_index("new_id").node
print("Населенных пунктов:", len(locations), " магазинов:", len(stores))

## Матрицы пространственных весов

Расстояния считаются в равновеликой конической проекции Альберса на эллипсоиде Красовского. Для
каждого населенного пункта берутся $k$ ближайших, связь учитывается в обе стороны, диагональ
нулевая, веса нормируются по строкам: $w_{rv} = a_{rv} / \sum_u a_{ru}$.

In [ ]:
from pyproj import Transformer

PROJECTION = ("+proj=aea +lat_1=52 +lat_2=64 +lat_0=0 +lon_0=105 +x_0=18500000 +y_0=0 "
              "+ellps=krass +units=m +towgs84=28,-130,-95,0,0,0,0 +no_defs")
transformer = Transformer.from_crs("EPSG:4326", PROJECTION, always_xy=True)
x_meters, y_meters = transformer.transform(locations.settlement_lon.to_numpy(), locations.settlement_lat.to_numpy())
projected = np.column_stack([x_meters, y_meters])
assert np.isfinite(projected).all()
distances = cdist(projected, projected)
np.fill_diagonal(distances, np.inf)
nearest = np.argsort(distances, axis=1, kind="stable")[:, :12]

weights = {}
for k in [6, 12]:
    adjacency = sparse.csr_matrix((np.ones(len(locations) * k),
                                   (np.repeat(np.arange(len(locations)), k), nearest[:, :k].ravel())),
                                  shape=(len(locations), len(locations)))
    adjacency = adjacency.maximum(adjacency.T)
    adjacency.setdiag(0)
    adjacency.eliminate_zeros()
    weight_matrix = (sparse.diags(1 / np.asarray(adjacency.sum(axis=1)).ravel()) @ adjacency).tocsr()
    assert np.all(weight_matrix.diagonal() == 0)
    np.testing.assert_allclose(np.asarray(weight_matrix.sum(axis=1)).ravel(), 1, atol=1e-12)
    weights["wx_knn" + str(k)] = weight_matrix
    print("kNN" + str(k), "- связей:", weight_matrix.nnz)

## Пространственные лаги признаков WX

Для числового признака сначала берется среднее по магазинам населенного пункта в месяце, затем
взвешенное среднее по соседним населенным пунктам, $(WX)_{rjt} = \sum_v w_{rv} \bar{x}_{vjt}$.
Магазины одного населенного пункта соседями друг для друга не считаются. Пропуски у соседей
сохраняются без заполнения. Идентификаторы, календарные и категориальные поля в $WX$ не входят.

In [ ]:
EXCLUDED_COLUMNS = {"new_id", "month", "prev_month", "year", "year_month_index", "days_lag_1", "days_in_month"}


def source_columns(x, categorical):
    return [c for c in x if c not in EXCLUDED_COLUMNS and c not in categorical and not c.startswith("is_")
            and not c.endswith(("_cb", "_cat", "_category")) and pd.api.types.is_numeric_dtype(x[c])
            and not pd.api.types.is_bool_dtype(x[c])]


def spatial_lags(x, meta, columns, weight_matrix):
    assert x.index.equals(meta.index)
    nodes = meta.new_id.map(store_nodes)
    assert nodes.notna().all()
    frame = x[columns].reset_index(drop=True).assign(node=nodes.to_numpy(), time=meta.year_month_index.to_numpy())
    grouped = frame.groupby(["time", "node"], sort=True)[columns].mean()
    result = np.full((len(frame), len(columns)), np.nan, dtype=np.float32)
    for t in np.sort(frame.time.unique()):
        part = grouped.loc[t].reindex(range(weight_matrix.shape[0]))
        lag = weight_matrix @ part.to_numpy(dtype=np.float64)
        positions = np.flatnonzero(frame.time.to_numpy() == t)
        result[positions] = lag[frame.node.to_numpy()[positions]]
    return pd.DataFrame(result, index=x.index, columns=["course_wx__" + c for c in columns])


def model_matrices(data, model):
    """Матрицы признаков исходного решения для обучения и прогноза."""
    features = globals()[model + "_features"]
    if model == "catboost_log_level":
        train, forecast = data["train_features"], data["val_features"]
        categorical = [c for c in features if (pd.api.types.is_object_dtype(train[c])
                       or pd.api.types.is_string_dtype(train[c]) or isinstance(train[c].dtype, pd.CategoricalDtype))]
        categorical += [c for c in ["new_id", "month", "prev_month"] if c in features and c not in categorical]
        x_train = _cb_prepare_X(train, features, categorical, name="full_train")
        x_forecast = _cb_prepare_X(forecast, features, categorical, name="march_forecast")
        y_train = make_forecast_target(train, mode="direct_log1p")
        return x_train, x_forecast, y_train, categorical
    mode = "growth_log" if model.endswith("log_diff") else "direct_log1p"
    pack = _prepare_numeric_model_pack(data, selected_features=features, target_mode=mode, label=model, verbose=False)
    validation = pack["validation"]
    return validation["X_train"][features], validation["X_val"][features], validation["y_train"], []


def prepare_model_data(data, model, configuration):
    x_train, x_forecast, y_train, categorical = model_matrices(data, model)
    columns = []
    if configuration != "baseline":
        columns = source_columns(x_train, categorical)
        weight_matrix = weights[configuration]
        x_train = pd.concat([x_train, spatial_lags(x_train, data["train_features"], columns, weight_matrix)], axis=1)
        x_forecast = pd.concat([x_forecast, spatial_lags(x_forecast, data["val_features"], columns, weight_matrix)], axis=1)
    return x_train, x_forecast, y_train, categorical, columns

## Полное обучение четырех моделей

Каждая модель обучается с исходными гиперпараметрами и полным числом деревьев без ранней
остановки. Прогнозы моделей на логарифме выручки возвращаются к исходной шкале через
$\exp(\hat{z}) - 1$, прогнозы приростов - через $y_{i,t-1} \exp(\hat{g}_{it})$.

In [ ]:
def fit_and_forecast(data, model, configuration):
    x_train, x_forecast, y_train, categorical, columns = prepare_model_data(data, model, configuration)
    params = dict(globals()[model + "_params"])
    count = TREE_COUNTS[model]
    if model == "catboost_log_level":
        from catboost import CatBoostRegressor, Pool
        params.update(iterations=count, thread_count=THREADS, allow_writing_files=False, verbose=500)
        estimator = CatBoostRegressor(**params)
        estimator.fit(Pool(x_train, y_train, cat_features=categorical))
        prediction = _cb_inverse_log1p(estimator.predict(x_forecast))
    elif model.startswith("lightgbm"):
        import lightgbm as lgb
        params = _clean_lgbm_params(params)
        params.update(n_estimators=count, n_jobs=THREADS, verbosity=-1)
        estimator = lgb.LGBMRegressor(**params)
        estimator.fit(x_train, y_train)
        prediction = inverse_forecast_target(estimator.predict(x_forecast), data["val_features"],
                                             mode="growth_log" if model.endswith("log_diff") else "direct_log1p")
    else:
        import xgboost as xgb
        params = _clean_xgb_params(params, keep_early_stopping=False)
        params.update(n_estimators=count, device=XGBOOST_DEVICE, n_jobs=THREADS)
        estimator = xgb.XGBRegressor(**params)
        valid = np.isfinite(np.asarray(y_train, dtype=float))
        estimator.fit(x_train.loc[valid], np.asarray(y_train)[valid], verbose=False)
        prediction = inverse_forecast_target(estimator.predict(x_forecast), data["val_features"], mode="growth_log")
    prediction = np.asarray(prediction, dtype=float)
    assert len(prediction) == 18657 and np.isfinite(prediction).all() and (prediction > 0).all()
    print(model, configuration, "- признаков:", x_train.shape[1], " из них WX:", len(columns))
    del x_train, x_forecast, estimator
    gc.collect()
    return prediction


predictions = {}
for model in MODELS:
    start = time.perf_counter()
    predictions[model] = fit_and_forecast(forecast_data, model, FINAL_CONFIGURATION[model])
    print(model, "обучена за", round((time.perf_counter() - start) / 60, 1), "мин")

## Объединение прогнозов

Правило объединения исходного решения сохранено. По четырем прогнозам считаются арифметическое
среднее $A$, логарифмическое среднее $L$ и два пороговых правила: $G$ заменяет среднее медианой,
когда $\max p_m / \min p_m > 1.08$, $H$ делает то же без LightGBM на приростах. Итоговый прогноз
равен логарифмическому среднему четырех кандидатов.

In [ ]:
matrix = np.column_stack([predictions[m] for m in MODELS])
arithmetic = matrix.mean(axis=1)
log_mean = np.expm1(np.log1p(np.maximum(matrix, 0)).mean(axis=1))
ratio = matrix.max(axis=1) / np.maximum(matrix.min(axis=1), 1e-9)
gated = np.where(ratio > 1.08, np.median(matrix, axis=1), arithmetic)
without_weak = matrix[:, [0, 2, 3]]
ratio_without_weak = without_weak.max(axis=1) / np.maximum(without_weak.min(axis=1), 1e-9)
gated_without_weak = np.where(ratio_without_weak > 1.08, np.median(without_weak, axis=1), without_weak.mean(axis=1))
candidates = np.column_stack([arithmetic, log_mean, gated, gated_without_weak])
final_forecast = np.expm1(np.log1p(np.maximum(candidates, 0)).mean(axis=1))

submission = pd.DataFrame({"new_id": forecast_data["val_features"].new_id.to_numpy(), "rto": final_forecast})
assert len(submission) == 18657 and submission.new_id.is_unique
submission.to_csv(SUBMISSION_PATH, index=False)
print("Прогноз сохранен:", SUBMISSION_PATH)
submission.describe()